Analysis notebook for *Intensity-Persistence Classes of Northwestern Pacific Tropical
Cyclones and the Limits of Unsupervised Rapid-Intensification Discovery:
A 48-Year Best-Track Analysis*.

## Requirements

```bash
pip install numpy pandas scipy scikit-learn hmmlearn joblib matplotlib tqdm
```

Model fitting uses `hmmlearn`, whose expectation-maximisation routines run on
CPU. The configuration grid is parallelised across cores with `joblib`.

## Results

Running all cells creates `results/` containing tabular output, the ten figures
in the order they appear in the paper (`Fig1.pdf` - `Fig10.pdf`), a summary of
all reported quantities, and a zip archive of the whole directory.

## Contents

| Section | Content | Paper |
|---|---|---|
| 1 | Configuration and environment | -- |
| 2 | Data, temporal grid, feature construction | Sec. 2.1--2.3 |
| 3 | Kalman filtering and smoothing | Sec. 2.4 |
| 4 | Quantisation structure | Sec. 3.1, Fig. 1 |
| 5--6 | Configuration grid | Sec. 3.2, Fig. 2 |
| 7 | Two-state wind-only model | Sec. 3.3, Tab. 1, Fig. 3 |
| 8 | Intensity threshold and conditional tests | Sec. 3.4--3.5, Tab. 2--3, Fig. 4 |
| 9 | Seasonal and inter-annual composition | Sec. 3.6, Fig. 5 |
| 10 | Case studies | Sec. 3.7, Fig. 6 |
| 11 | Environmental separation | Sec. 3.8, Tab. 5, Fig. 7 |
| 12 | Higher-state structure | Sec. 3.9, Fig. 8 |
| 13 | Precursor state distribution | Sec. 3.10, Tab. 4, Fig. 9 |
| 14 | Supervised baselines | Sec. 4.3, Tab. 6, Fig. 10 |
| 15 | Epoch sensitivity | Sec. 4.8 |
| 16 | Supplementary diagnostics quoted in the text | Sec. 2.1, 3.2, 3.8 |
| 17 | Summary and archive | -- |


## 1. Configuration and environment

In [1]:
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[_v] = "1"

import sys, json, time, platform, warnings, logging, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats as sps
import matplotlib
matplotlib.use("Agg")
import matplotlib.ticker
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import GroupKFold
from hmmlearn import hmm as hmmlearn_hmm
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")
logging.getLogger("hmmlearn").setLevel(logging.ERROR)

# ----------------------------- input files ----------------------------------
CSV_PARTS = ["project_dataset_part1.csv",
             "project_dataset_part2.csv"]
OUT = Path("results"); OUT.mkdir(exist_ok=True)
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)

# ----------------------------- analysis settings ----------------------------
MIN_WIND_KT    = 35          # JMA reports no wind below tropical-storm strength
TRAIN_YEAR_MAX = 2014
VALID_YEAR_MAX = 2019
RI_THRESH      = 30.0        # kt per 24 h
MIN_SEQ_LEN    = 5           # minimum track length for the sequence decoder
SEED           = 42

# Observation-noise variance of a uniform quantisation error on a 5-kt lattice.
Q_PROC, R_OBS  = 1.0, 25.0 / 12.0

# ----------------------------- grid settings --------------------------------
# Two cores are held back so the machine stays responsive during the grid.
N_JOBS     = max(1, (os.cpu_count() or 4) - 2)
GRID_K     = list(range(2, 10))
GRID_COV   = ["diag", "full"]
GRID_SEEDS = [0, 1, 2]
RESTART_S1, ITER_S1 = 15, 80     # stage 1: many starts, few iterations
RESTART_S2, ITER_S2 = 3, 400     # stage 2: refine the three best

N_BOOT_STORM = 2000              # storm-level bootstrap replicates
N_PERM       = 50_000            # exact stratified permutation draws
KAPPA_STABLE = 1e4               # numerical-stability threshold on condition number
KAPPA_CAP    = 1e18              # display cap; beyond this double precision is empty

# ----------------------------- plotting -------------------------------------
DPI = 300
COL_HIGH, COL_LOW = "#8E44AD", "#7F8C8D"
COL_RAW, COL_FILT = "#C0392B", "#27AE60"
plt.rcParams.update({"font.family": "serif", "font.size": 11, "axes.titlesize": 12,
                     "axes.labelsize": 11, "legend.fontsize": 9, "xtick.labelsize": 10,
                     "ytick.labelsize": 10, "axes.grid": True, "grid.alpha": 0.25,
                     "figure.dpi": 100})

# Display names used in every figure and exported table.
FS_LABEL = {"F_wind":           "Wind only (1-D)",
            "F4D_overlap":      "4-D overlapping \u0394V",
            "F2D_nonoverlap":   "2-D non-overlapping \u0394V",
            "F_filt":           "Filtered wind + \u0394V24",
            "F_filt_env":       "Filtered wind + environment",
            "F_filt_env_conv":  "Filtered wind + environment + convective"}
COV_LABEL = {"diag": "diagonal", "full": "full"}

ENV_INFO = dict(python=sys.version.split()[0], platform=platform.platform(),
                numpy=np.__version__, pandas=pd.__version__,
                sklearn=__import__("sklearn").__version__,
                hmmlearn=__import__("hmmlearn").__version__,
                n_cpu=os.cpu_count(), n_jobs=N_JOBS, seed=SEED,
                Q=Q_PROC, R=R_OBS, timestamp=time.strftime("%Y-%m-%d %H:%M:%S"))
json.dump(ENV_INFO, open(OUT / "00_environment.json", "w"), indent=2)
print(json.dumps(ENV_INFO, indent=2))


{
  "python": "3.10.19",
  "platform": "Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39",
  "numpy": "1.26.4",
  "pandas": "2.2.3",
  "sklearn": "1.6.1",
  "hmmlearn": "0.3.3",
  "n_cpu": 24,
  "n_jobs": 22,
  "seed": 42,
  "Q": 1.0,
  "R": 2.0833333333333335,
  "timestamp": "2026-10-23 00:23:34"
}


## 2. Data, temporal grid and feature construction

In [2]:
USE_COLS = ["Storm_Name", "International_ID", "Year", "time", "MaxWind", "Lat", "Lon",
            "sst_C", "u200", "v200", "u850", "v850", "so26chgt", "t200",
            "tp", "r850", "r700", "r500", "vo850", "sohtc300", "somxl030", "V_DLM"]

raw = pd.concat([pd.read_csv(f, usecols=lambda c: c in USE_COLS, low_memory=False)
                 for f in CSV_PARTS], ignore_index=True)
raw["time"] = pd.to_datetime(raw["time"], errors="coerce", dayfirst=True)
raw = raw.dropna(subset=["time"])
raw["year"]  = raw["time"].dt.year
raw["month"] = raw["time"].dt.month
raw["storm_id"]   = raw["International_ID"].astype(str) + "_" + raw["Year"].astype(str)
raw["storm_name"] = raw.get("Storm_Name", raw["storm_id"])
raw.loc[raw["sst_C"] < 5, "sst_C"] = np.nan

n_before = len(raw)
raw = (raw.sort_values(["storm_id", "time"])
          .drop_duplicates(["storm_id", "time"], keep="first"))
n_dup = n_before - len(raw)

df = (raw[raw["MaxWind"].notna() & (raw["MaxWind"] >= MIN_WIND_KT)]
        .sort_values(["storm_id", "time"]).reset_index(drop=True).copy())

DIAG = {"rows_raw": int(n_before), "rows_dup_dropped": int(n_dup),
        "obs_working": int(len(df)), "storms_working": int(df.storm_id.nunique()),
        "min_MaxWind_present": float(df.MaxWind.min()),
        "MaxWind_on_5kt_lattice": bool((df.MaxWind % 5 == 0).all())}

# --------------------------- temporal grid ----------------------------------
dt_h = df.groupby("storm_id")["time"].diff().dt.total_seconds() / 3600.0
share = (dt_h.dropna().value_counts(normalize=True) * 100).round(3)
DIAG["timestep_pct"] = {str(k): float(v) for k, v in share.head(10).items()}
DIAG["pct_step_not_6h"] = float(100 * ((dt_h.notna()) & (abs(dt_h - 6) > 1e-6)).sum()
                                / max(dt_h.notna().sum(), 1))
print(f"observations {len(df):,}   storms {df.storm_id.nunique():,}   "
      f"duplicate timestamps removed {n_dup:,}")
print(f"time steps other than 6 h: {DIAG['pct_step_not_6h']:.2f}%")
print(share.head(6).to_string())

# --------------------------- timestamp-matched lags -------------------------
def val_at_offset(frame, hours, col="MaxWind", tol_min=1):
    """Value of `col` at exactly (time - hours); negative `hours` looks ahead."""
    left = frame[["storm_id", "time"]].copy()
    left["_ix"] = np.arange(len(frame))
    right = frame[["storm_id", "time", col]].copy()
    right["time"] = right["time"] + pd.Timedelta(hours=hours)
    merged = pd.merge_asof(left.sort_values("time"), right.sort_values("time"),
                           on="time", by="storm_id",
                           tolerance=pd.Timedelta(minutes=tol_min), direction="nearest")
    return merged.set_index("_ix")[col].reindex(np.arange(len(frame))).values

for hours, name in [(6, "V_m6"), (12, "V_m12"), (24, "V_m24"), (-24, "V_p24")]:
    df[name] = val_at_offset(df, hours)

df["dV_6h"]         = df["MaxWind"] - df["V_m6"]
df["dV_12h"]        = df["MaxWind"] - df["V_m12"]
df["dV_24h"]        = df["MaxWind"] - df["V_m24"]
df["dV_12_24"]      = df["V_m12"] - df["V_m24"]        # non-overlapping difference
df["dV_24h_future"] = df["V_p24"] - df["MaxWind"]
df["RI"] = np.where(df["dV_24h_future"].notna(),
                    (df["dV_24h_future"] >= RI_THRESH).astype(float), np.nan)

# Row-offset differencing is retained as a diagnostic only.
row_shift = df.groupby("storm_id")["MaxWind"].shift(-4) - df["MaxWind"]
has_row, has_time = row_shift.notna(), df.dV_24h_future.notna()
DIAG.update(label_rowshift_n=int(has_row.sum()),
            label_rowshift_base=float(100 * (row_shift[has_row] >= RI_THRESH).mean()),
            label_timebased_n=int(has_time.sum()),
            label_timebased_base=float(100 * (df.loc[has_time, "dV_24h_future"] >= RI_THRESH).mean()),
            pct_dV24_differs=float(100 * (row_shift[has_row & has_time]
                                          != df.loc[has_row & has_time, "dV_24h_future"]).mean()),
            n_RI_rowshift=int((row_shift >= RI_THRESH).sum()),
            n_RI_timebased=int((df.dV_24h_future >= RI_THRESH).sum()))
print(f"\nRI labels, row offset     : n={DIAG['label_rowshift_n']:,}  "
      f"base={DIAG['label_rowshift_base']:.3f}%  ({DIAG['n_RI_rowshift']} cases)")
print(f"RI labels, timestamp match: n={DIAG['label_timebased_n']:,}  "
      f"base={DIAG['label_timebased_base']:.3f}%  ({DIAG['n_RI_timebased']} cases)  <- used")
print(f"delta-V24 differs for {DIAG['pct_dV24_differs']:.1f}% of records defined under both")

# --------------------------- derived environment ----------------------------
du = df["u200"] - df["u850"]
dv = df["v200"] - df["v850"]
df["shear"] = np.sqrt(du ** 2 + dv ** 2) * 1.944          # m/s -> kt

Ts = df["sst_C"] + 273.15
def mpi_kt(sst_k, t_out_k):
    return np.sqrt(np.maximum(0.9 * sst_k * np.maximum(sst_k - t_out_k, 0) / t_out_k, 0)) * 1.944
df["T_out_param"] = 210.0 - 0.5 * np.abs(df["Lat"])       # latitude parametrisation
df["T_out_obs"]   = df["t200"] if "t200" in df.columns else np.nan
df["mpi_gap"]     = mpi_kt(Ts, df["T_out_param"]) - df["MaxWind"]
df["mpi_gap_obs"] = mpi_kt(Ts, df["T_out_obs"]) - df["MaxWind"]
DIAG["corr_mpi_gap_variants"] = float(df[["mpi_gap", "mpi_gap_obs"]].corr().iloc[0, 1])

df["split"] = np.where(df.year <= TRAIN_YEAR_MAX, "train",
              np.where(df.year <= VALID_YEAR_MAX, "valid", "test"))
DIAG.update(n_train=int((df.split == "train").sum()),
            n_valid=int((df.split == "valid").sum()),
            n_test=int((df.split == "test").sum()))
print(f"\ntrain {DIAG['n_train']:,} | valid {DIAG['n_valid']:,} | test {DIAG['n_test']:,}")
json.dump(DIAG, open(OUT / "01_data_diagnostics.json", "w"), indent=2, default=str)


observations 28,156   storms 1,220   duplicate timestamps removed 92
time steps other than 6 h: 19.16%
time
6.0     80.836
3.0     17.909
1.0      0.601
2.0      0.531
30.0     0.022
18.0     0.019

RI labels, row offset     : n=23,338  base=3.072%  (717 cases)
RI labels, timestamp match: n=21,876  base=3.364%  (736 cases)  <- used
delta-V24 differs for 12.3% of records defined under both

train 22,580 | valid 3,196 | test 2,380


## 3. Kalman filtering and smoothing

In [3]:
def kalman_1d(y, Q, R, P0=100.0):
    """Return (RTS-smoothed, forward-filtered) estimates of a scalar series."""
    n = len(y)
    x_pred = np.empty(n); P_pred = np.empty(n)
    x_filt = np.empty(n); P_filt = np.empty(n)
    x_pred[0], P_pred[0] = y[0], P0
    gain = P_pred[0] / (P_pred[0] + R)
    x_filt[0] = x_pred[0] + gain * (y[0] - x_pred[0])
    P_filt[0] = (1 - gain) * P_pred[0]
    for t in range(1, n):
        x_pred[t] = x_filt[t - 1]
        P_pred[t] = P_filt[t - 1] + Q
        gain = P_pred[t] / (P_pred[t] + R)
        x_filt[t] = x_pred[t] + gain * (y[t] - x_pred[t])
        P_filt[t] = (1 - gain) * P_pred[t]
    x_smooth = x_filt.copy()
    for t in range(n - 2, -1, -1):
        A = P_filt[t] / P_pred[t + 1]
        x_smooth[t] = x_filt[t] + A * (x_smooth[t + 1] - x_pred[t + 1])
    return x_smooth, x_filt

def apply_kalman(frame, Q, R):
    smooth = np.empty(len(frame)); filt = np.empty(len(frame))
    for _, idx in frame.groupby("storm_id").groups.items():
        idx = np.asarray(idx)
        a, b = kalman_1d(frame["MaxWind"].values[idx].astype(float), Q, R)
        smooth[idx], filt[idx] = a, b
    return smooth, filt

df["V_smooth"], df["V_filt"] = apply_kalman(df, Q_PROC, R_OBS)

for source, tag in [("V_smooth", "s"), ("V_filt", "f")]:
    df[f"dV_24h_{tag}"] = df[source] - val_at_offset(df, 24, col=source)

# ---- sensitivity of the near-zero fraction and of leakage to Q and R --------
rows = []
for Q in [0.25, 0.5, 1.0, 2.0, 4.0]:
    for R, tag in [(25 / 12, "5^2/12 (quantisation variance)"),
                   (6.25, "2.5^2 (half bin width)"),
                   (1.0, "1.0 (reference)")]:
        sm, ft = apply_kalman(df, Q, R)
        d6 = pd.Series(sm) - pd.Series(val_at_offset(df.assign(_t=sm), 6, col="_t"))
        defined = df.dV_24h_future.notna()
        rows.append(dict(Q=Q, R=round(R, 4), R_label=tag,
                         frac_near_zero_dV6=float(100 * (d6.abs() < 0.5).mean()),
                         corr_smoother_residual_future=float(np.corrcoef(
                             (sm - df.MaxWind.values)[defined], df.dV_24h_future[defined])[0, 1]),
                         corr_filter_residual_future=float(np.corrcoef(
                             (ft - df.MaxWind.values)[defined], df.dV_24h_future[defined])[0, 1])))
sens = pd.DataFrame(rows)
sens.to_csv(OUT / "02_kalman_sensitivity.csv", index=False)
print(sens.round(4).to_string(index=False))
print("\nA large correlation between the smoother residual and the forward 24-h")
print("change indicates that the smoother carries verification-window information.")


   Q      R                        R_label  frac_near_zero_dV6  corr_smoother_residual_future  corr_filter_residual_future
0.25 2.0833 5^2/12 (quantisation variance)             19.0013                         0.6292                      -0.3051
0.25 6.2500         2.5^2 (half bin width)             28.2782                         0.6311                      -0.2122
0.25 1.0000                1.0 (reference)             16.9200                         0.5998                      -0.3501
0.50 2.0833 5^2/12 (quantisation variance)             16.9520                         0.6020                      -0.3482
0.50 6.2500         2.5^2 (half bin width)             21.6792                         0.6338                      -0.2741
0.50 1.0000                1.0 (reference)             17.0337                         0.5461                      -0.3770
1.00 2.0833 5^2/12 (quantisation variance)             16.9626                         0.5496                      -0.3760
1.00 6.2500     

## 4. Quantisation structure — Figure 1

In [4]:
def make_seqs(frame, cols, scaler):
    """Group observations into per-storm sequences of at least MIN_SEQ_LEN."""
    X = scaler.transform(frame[cols].values)
    seqs, lengths, index_sets = [], [], []
    for _sid, idx in frame.groupby("storm_id", sort=False).indices.items():
        if len(idx) >= MIN_SEQ_LEN:
            idx = np.sort(np.asarray(idx))
            seqs.append(X[idx]); lengths.append(len(idx)); index_sets.append(idx)
    empty = np.zeros((0, len(cols)))
    return (np.concatenate(seqs) if seqs else empty), lengths, index_sets

QUANT = {}
d6, d24 = df["dV_6h"].dropna(), df["dV_24h"].dropna()
QUANT["pct_dV6_exact_zero"]  = float(100 * (d6 == 0).mean())
QUANT["pct_dV24_exact_zero"] = float(100 * (d24 == 0).mean())
for source, tag in [("V_smooth", "smoother"), ("V_filt", "filter")]:
    dd = pd.Series(df[source].values - val_at_offset(df, 6, col=source)).dropna()
    QUANT[f"pct_dV6_near_zero_{tag}"] = float(100 * (dd.abs() < 0.5).mean())
print(json.dumps(QUANT, indent=2))

# ---- representative K = 6 fit on overlapping differences -------------------
F4D = ["MaxWind", "dV_6h", "dV_12h", "dV_24h"]
sub = df.dropna(subset=F4D).copy()
scaler6 = StandardScaler().fit(sub.loc[sub.split == "train", F4D].values)
Xtr6, ltr6, _ = make_seqs(sub[sub.split == "train"], F4D, scaler6)
model6, cov6 = None, None
for cov in ["full", "diag"]:
    try:
        m = hmmlearn_hmm.GaussianHMM(n_components=6, covariance_type=cov,
                                     n_iter=ITER_S2, tol=1e-4, random_state=SEED)
        m.fit(Xtr6, ltr6); model6, cov6 = m, cov; break
    except Exception as exc:
        print(f"  K=6 {cov} covariance did not fit: {exc}")

Xall6, _, idx6 = make_seqs(sub, F4D, scaler6)
states6 = np.full(len(sub), -1, int); pos = 0
for idx in idx6:
    states6[idx] = model6.predict(Xall6[pos:pos + len(idx)]); pos += len(idx)
sub["state6"] = states6

zero_tab = (sub[sub.state6 >= 0].groupby("state6")
              .agg(n=("MaxWind", "size"),
                   frac_zero=("dV_6h", lambda x: float((x == 0).mean())),
                   mean_V=("MaxWind", "mean"), mean_dV6=("dV_6h", "mean"))
              .reset_index().sort_values("mean_dV6", ascending=False)
              .reset_index(drop=True))
# Display labels S1..S6 follow the ordering by mean 6-h intensity change
# (most intensifying first); the raw hmmlearn index is kept for reference.
zero_tab.insert(0, "label", [f"S{i + 1}" for i in range(len(zero_tab))])
zero_tab = zero_tab.rename(columns={"state6": "hmm_index"})
zero_tab.to_csv(OUT / "03_quantisation_states.csv", index=False)
QUANT["k6_covariance"] = cov6
QUANT["k6_states_fully_absorbing"] = int((zero_tab.frac_zero > 0.99).sum())
QUANT["k6_pct_obs_in_absorbing"] = float(
    100 * zero_tab.loc[zero_tab.frac_zero > 0.99, "n"].sum() / zero_tab.n.sum())
QUANT["k6_max_frac_zero"] = float(zero_tab.frac_zero.max())
QUANT["k6_n_obs_decoded"] = int(zero_tab.n.sum())
largest = zero_tab.loc[zero_tab.n.idxmax()]
QUANT["k6_largest_state_n"] = int(largest.n)
QUANT["k6_largest_state_frac_zero"] = float(largest.frac_zero)
print(zero_tab.round(3).to_string(index=False))
json.dump(QUANT, open(OUT / "03_quantisation.json", "w"), indent=2)

# ------------------------------- Figure 1 -----------------------------------
# Larger fonts for publication
plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11.5
})

# Slightly larger canvas to accommodate the larger fonts
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
bins = np.arange(30, 150, 1)

ax[0].hist(df.MaxWind.dropna(), bins=bins, color=COL_RAW, alpha=0.8,
           label="Raw (5-kt lattice)")
ax[0].hist(df.V_filt.dropna(), bins=bins, color=COL_FILT, alpha=0.7,
           label=f"Causal Kalman filter\n($Q$={Q_PROC} kt$^2$, $R$={R_OBS:.3f} kt$^2$)")
ax[0].set_xlabel("Maximum sustained wind (kt)")
ax[0].set_ylabel("Number of observations")
ax[0].set_title("(a) Best-track intensity quantisation")
ax[0].legend(loc="upper right", fontsize=11.5)

bar_colours = ["#8E44AD" if f > 0.99 else ("#D7BDE2" if f > 0.5 else "#BDC3C7")
               for f in zero_tab.frac_zero]
ax[1].bar(range(len(zero_tab)), zero_tab.frac_zero * 100, color=bar_colours,
          edgecolor="black", linewidth=0.5)
ax[1].set_xticks(range(len(zero_tab)))
ax[1].set_xticklabels([f"{lab}\n{dv:+.1f}" for lab, dv in
                       zip(zero_tab.label, zero_tab.mean_dV6)], fontsize=12)
ax[1].set_ylabel(r"Observations with $\Delta V_{6h}=0$ exactly (%)")
ax[1].set_xlabel(r"Hidden state, ordered by mean $\Delta V_{6h}$ (kt per 6 h)"
                 f"\n($K$=6, 4-D overlapping features, {cov6} covariance)")
ax[1].set_title("(b) Quantisation-driven state absorption")
ax[1].set_ylim(0, 112)

for i, n in enumerate(zero_tab.n):
    ax[1].text(i, zero_tab.frac_zero.iloc[i] * 100 + 3, f"n={int(n):,}",
               ha="center", fontsize=11)

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(FIG / f"Fig1.{ext}", dpi=DPI, bbox_inches="tight")
plt.close()
print("-> Fig1")

{
  "pct_dV6_exact_zero": 52.15249355767774,
  "pct_dV24_exact_zero": 17.81861400621686,
  "pct_dV6_near_zero_smoother": 18.099135970895862,
  "pct_dV6_near_zero_filter": 22.930877671668938
}
label  hmm_index     n  frac_zero  mean_V  mean_dV6
   S1          1  1059      0.006  76.313     8.404
   S2          4  1595      0.000  68.182     5.000
   S3          2  4300      1.000  64.881     0.000
   S4          5 11296      0.608  63.171    -0.385
   S5          3  2050      0.000  58.400    -6.098
   S6          0  1205      0.018  63.212    -6.249
-> Fig1


## 5. Configuration grid: definition

Six feature sets are crossed with eight hidden-state counts, two covariance
structures and three random seeds, giving 288 configurations. Each fit uses a
two-stage restart protocol: fifteen short starts, then refinement of the three
best.

For each configuration the cell records the per-state RI proportion, the
maximum emission-covariance condition number, the training and validation
log-likelihoods, and the Bayesian information criterion.

In [5]:
FEATURE_SETS = {
    "F_wind":          ["MaxWind"],
    "F4D_overlap":     ["MaxWind", "dV_6h", "dV_12h", "dV_24h"],
    "F2D_nonoverlap":  ["MaxWind", "dV_12_24"],
    "F_filt":          ["V_filt", "dV_24h_f"],
    "F_filt_env":      ["V_filt", "dV_24h_f", "sst_C", "shear", "so26chgt", "mpi_gap_obs"],
    "F_filt_env_conv": ["V_filt", "dV_24h_f", "sst_C", "shear", "so26chgt", "mpi_gap_obs",
                        "tp", "r700", "r500", "vo850", "sohtc300"],
}
FEATURE_SETS = {k: [c for c in v if c in df.columns] for k, v in FEATURE_SETS.items()}
for name, cols in FEATURE_SETS.items():
    print(f"  {name:17s} {len(cols)}-D  {cols}")

def n_params(K, n_features, cov):
    n_cov = n_features if cov == "diag" else n_features * (n_features + 1) // 2
    return (K - 1) + K * (K - 1) + K * n_features + K * n_cov

def prepare(cols):
    """Scale on the training partition and build sequence arrays."""
    sub = df.dropna(subset=cols).copy()
    scaler = StandardScaler().fit(sub.loc[sub.split == "train", cols].values)
    Xtr, ltr, _ = make_seqs(sub[sub.split == "train"], cols, scaler)
    Xva, lva, _ = make_seqs(sub[sub.split == "valid"], cols, scaler)
    Xall, _, idx_sets = make_seqs(sub, cols, scaler)
    order = np.concatenate(idx_sets) if idx_sets else np.array([], int)
    ri = np.full(len(sub), np.nan)
    ri[sub.RI.notna().values] = sub.RI.dropna().values
    ri_ordered = ri[order]
    blocks, pos = [], 0
    for idx in idx_sets:
        blocks.append(np.arange(pos, pos + len(idx))); pos += len(idx)
    return dict(Xtr=Xtr, ltr=ltr, Xva=Xva, lva=lva, Xall=Xall, blocks=blocks,
                ri=np.nan_to_num(ri_ordered), defined=np.isfinite(ri_ordered))

def fit_one(fs_name, cols, K, cov, seed, data):
    """Fit one configuration and summarise it."""
    started = time.time()
    out = dict(feature_set=fs_name, n_features=len(cols), K=K, cov=cov, seed=seed,
               status="ok", error="")
    try:
        rng = np.random.RandomState(1000 * seed + K)
        stage1 = []
        for _ in range(RESTART_S1):
            try:
                m = hmmlearn_hmm.GaussianHMM(n_components=K, covariance_type=cov,
                                             n_iter=ITER_S1, tol=1e-4,
                                             random_state=rng.randint(1_000_000))
                m.fit(data["Xtr"], data["ltr"])
                stage1.append((m.score(data["Xtr"], data["ltr"]), m))
            except Exception:
                pass
        if not stage1:
            out.update(status="fail", error="no stage-1 start converged")
            return out
        stage1.sort(key=lambda z: -z[0])

        best_ll, best = -np.inf, None
        for _, start in stage1[:RESTART_S2]:
            try:
                m = hmmlearn_hmm.GaussianHMM(n_components=K, covariance_type=cov,
                                             n_iter=ITER_S2, tol=1e-5,
                                             random_state=seed, init_params="")
                m.startprob_, m.transmat_ = start.startprob_.copy(), start.transmat_.copy()
                m.means_ = start.means_.copy()
                m.covars_ = (np.array([np.diag(c) for c in start.covars_])
                             if cov == "diag" else start.covars_.copy())
                m.fit(data["Xtr"], data["ltr"])
                ll = m.score(data["Xtr"], data["ltr"])
                if ll > best_ll:
                    best_ll, best = ll, m
            except Exception:
                pass
        if best is None:
            best_ll, best = stage1[0]

        conds = []
        for k in range(K):
            ev = np.linalg.eigvalsh(best.covars_[k])
            conds.append(ev.max() / max(ev.min(), 1e-300))
        out["max_cond"] = float(np.max(conds))
        out["med_cond"] = float(np.median(conds))

        states = np.full(data["Xall"].shape[0], -1, int); pos = 0
        for block in data["blocks"]:
            states[block] = best.predict(data["Xall"][pos:pos + len(block)])
            pos += len(block)

        pis, sizes = [], []
        for k in range(K):
            mask = (states == k) & data["defined"]
            sizes.append(int((states == k).sum()))
            pis.append(float(data["ri"][mask].mean()) if mask.sum() else np.nan)
        pis = np.asarray(pis, dtype=float)
        finite = np.isfinite(pis).any()
        out["max_pi_k"]    = float(np.nanmax(pis)) if finite else np.nan
        out["max_pi_k_n"]  = int(sizes[int(np.nanargmax(pis))]) if finite else 0
        out["min_pi_k"]    = float(np.nanmin(pis)) if finite else np.nan
        out["pi_k_all"]    = json.dumps([None if not np.isfinite(p) else round(p, 5) for p in pis])
        out["state_sizes"] = json.dumps(sizes)
        out["ll_train"]    = float(best_ll)
        out["ll_valid"]    = float(best.score(data["Xva"], data["lva"])) if data["Xva"].size else np.nan
        out["n_params"]    = int(n_params(K, len(cols), cov))
        out["bic"]         = float(-2 * best_ll + out["n_params"] * np.log(data["Xtr"].shape[0]))
    except Exception as exc:
        out.update(status="fail", error=str(exc)[:200])
    out["seconds"] = round(time.time() - started, 2)
    return out

PREPARED = {name: prepare(cols) for name, cols in FEATURE_SETS.items()}
JOBS = [(fs, FEATURE_SETS[fs], K, cov, sd)
        for fs in FEATURE_SETS for K in GRID_K for cov in GRID_COV for sd in GRID_SEEDS]
print(f"\nconfigurations to fit: {len(JOBS):,} on {N_JOBS} workers")


  F_wind            1-D  ['MaxWind']
  F4D_overlap       4-D  ['MaxWind', 'dV_6h', 'dV_12h', 'dV_24h']
  F2D_nonoverlap    2-D  ['MaxWind', 'dV_12_24']
  F_filt            2-D  ['V_filt', 'dV_24h_f']
  F_filt_env        6-D  ['V_filt', 'dV_24h_f', 'sst_C', 'shear', 'so26chgt', 'mpi_gap_obs']
  F_filt_env_conv   11-D  ['V_filt', 'dV_24h_f', 'sst_C', 'shear', 'so26chgt', 'mpi_gap_obs', 'tp', 'r700', 'r500', 'vo850', 'sohtc300']

configurations to fit: 288 on 22 workers


## 6. Configuration grid: execution and Figure 2

In [6]:
started = time.time()
records = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
    delayed(fit_one)(fs, cols, K, cov, sd, PREPARED[fs]) for fs, cols, K, cov, sd in JOBS)
grid = pd.DataFrame(records)
grid["feature_set_label"]  = grid.feature_set.map(FS_LABEL)
grid["cov_label"]          = grid["cov"].map(COV_LABEL)
grid["numerically_stable"] = grid.max_cond < KAPPA_STABLE
grid.to_csv(OUT / "04_config_grid_full.csv", index=False)
print(f"\nfinished in {(time.time() - started) / 60:.1f} min; "
      f"{(grid.status == 'ok').sum()}/{len(grid)} converged")

ok = grid[grid.status == "ok"].copy()
ok["max_pi_pct"] = 100 * ok.max_pi_k
base_rate = DIAG["label_timebased_base"]
pi_stable_max = float(ok.loc[ok.max_cond < KAPPA_STABLE, "max_pi_pct"].max())
n_over10 = int((ok.max_pi_pct >= 10).sum())
n_over10_singular = int(((ok.max_pi_pct >= 10) & (ok.max_cond >= KAPPA_STABLE)).sum())

print("\nlargest per-state RI proportion (%), by feature set and covariance")
print(ok.pivot_table(index="feature_set", columns="cov",
                     values="max_pi_pct", aggfunc="max").round(2).to_string())
print(f"\ngrid maximum            {ok.max_pi_pct.max():.2f}%   (base rate {base_rate:.2f}%)")
print(f"numerically stable max  {pi_stable_max:.2f}%")
print(f"configurations >= 10%   {n_over10} of {len(ok)}, of which {n_over10_singular} singular")
top = ok.loc[ok.max_pi_pct.idxmax()]
print(f"grid maximum attained by {top.feature_set}, K={top.K}, {top['cov']} covariance, "
      f"seed {top.seed}, state size {top.max_pi_k_n:,}, kappa {top.max_cond:.2e}")
print("\nspread across seeds (sd of the maximum, percentage points)")
print(ok.groupby(["feature_set", "cov", "K"]).max_pi_pct.std()
        .groupby(level=[0, 1]).max().round(3).to_string())

# ------------------------------- Figure 2 -----------------------------------
# Larger fonts for publication
plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11.6
})

agg = (ok.groupby(["feature_set", "cov", "K"])
         .agg(pi_mean=("max_pi_pct", "mean"), pi_sd=("max_pi_pct", "std"),
              cond_max=("max_cond", "max")).reset_index())
agg["pi_sd"] = agg.pi_sd.fillna(0.0)
agg.to_csv(OUT / "04_config_grid_agg.csv", index=False)

SHOW = [("F4D_overlap", "full", "#7F8C8D", "s"),
        ("F2D_nonoverlap", "diag", "#E67E22", "o"),
        ("F_filt_env", "diag", "#8E44AD", "^"),
        ("F_filt_env_conv", "diag", "#16A085", "D")]
SHOW = [t for t in SHOW if ((agg.feature_set == t[0]) & (agg["cov"] == t[1])).any()]
if not SHOW:
    raise RuntimeError("no series selected for Figure 2; check feature-set names")

# Slightly larger canvas to accommodate the legend below the panels
fig, ax = plt.subplots(1, 2, figsize=(13, 5.8))
for fs, cov, colour, marker in SHOW:
    s = agg[(agg.feature_set == fs) & (agg["cov"] == cov)].sort_values("K")
    stable = (s.cond_max < KAPPA_STABLE).values
    ax[0].errorbar(s.K, s.pi_mean, yerr=s.pi_sd, color=colour, lw=1.7,
                   capsize=3, marker="", zorder=2, alpha=0.9)
    ax[0].scatter(s.K[stable], s.pi_mean[stable], marker=marker, s=62,
                  facecolor=colour, edgecolor=colour, zorder=3)
    ax[0].scatter(s.K[~stable], s.pi_mean[~stable], marker=marker, s=62,
                  facecolor="white", edgecolor=colour, linewidth=1.6, zorder=3)
    # Values near 1e300 overflow the log locator, so cap before plotting.
    ax[1].plot(s.K, np.minimum(s.cond_max.values, KAPPA_CAP), marker=marker,
               ms=7, lw=1.7, color=colour, zorder=3)
    off_scale = s.cond_max.values > KAPPA_CAP
    if off_scale.any():
        ax[1].scatter(s.K[off_scale], np.full(off_scale.sum(), KAPPA_CAP), marker="^",
                      s=80, facecolor=colour, edgecolor="black", linewidth=0.8, zorder=4)

ax[0].axhline(30, color="red", ls="--", lw=1.5)
ax[0].axhline(pi_stable_max, color="black", ls="-.", lw=1.3)
ax[0].axhline(base_rate, color="grey", ls=":", lw=1.3)

ax[0].annotate("RI-state threshold (30%)", xy=(9.30, 30.0), xytext=(9.30, 30.6),
               ha="right", va="bottom", fontsize=11.5, color="red")
ax[0].annotate(f"Max among numerically stable runs ({pi_stable_max:.1f}%)",
               xy=(5.0, pi_stable_max), xytext=(2.55, 15.2), ha="left", va="bottom",
               fontsize=11.5, arrowprops=dict(arrowstyle="->", lw=0.9, color="black",
                                             shrinkA=1, shrinkB=2))
ax[0].annotate(f"Base rate ({base_rate:.2f}%)", xy=(9.30, base_rate),
               xytext=(9.30, base_rate - 0.55), ha="right", va="top",
               fontsize=11.5, color="dimgrey")
ax[0].set_xlabel("Number of hidden states $K$")
ax[0].set_ylabel(r"max$_k\ \pi_k$ (%)")
ax[0].set_title("(a) No configuration discovers an RI state")
ax[0].set_ylim(0, 33); ax[0].set_xlim(1.6, 9.4)

ax[1].set_yscale("log")
ax[1].set_ylim(0.5, KAPPA_CAP * 100)
ax[1].axhline(KAPPA_STABLE, color="red", ls="--", lw=1.5)
ax[1].axhspan(1e16, KAPPA_CAP * 100, color="red", alpha=0.07, lw=0)

ax[1].annotate(r"Stability threshold ($\kappa=10^4$)", xy=(9.30, KAPPA_STABLE),
               xytext=(9.30, KAPPA_STABLE / 3.2), ha="right", va="top",
               fontsize=11.5, color="red")
ax[1].annotate(r"Beyond double precision ($\kappa>10^{16}$)", xy=(1.72, 1.2e19),
               xytext=(1.72, 1.2e19), ha="left", va="center",
               fontsize=11.5, color="firebrick")
ax[1].set_xlabel("Number of hidden states $K$")
ax[1].set_ylabel(r"Max covariance condition number $\kappa$")
ax[1].set_title("(b) Numerical stability deteriorates with $K$")
ax[1].set_xlim(1.6, 9.4)
ax[1].yaxis.set_major_locator(matplotlib.ticker.LogLocator(base=10, numticks=12))

handles = [Line2D([], [], color=c, marker=m, ms=8, lw=1.7, linestyle="-",
                  label=f"{FS_LABEL[fs]} ({COV_LABEL[cov]} cov.)")
           for fs, cov, c, m in SHOW]
handles += [Line2D([], [], color="black", marker="o", ms=8, lw=0, markerfacecolor="white",
                    markeredgewidth=1.6, label=r"Open symbol: $\kappa \geq 10^4$ (singular)"),
            Line2D([], [], color="black", marker="^", ms=8, lw=0, markeredgecolor="black",
                    label=r"Filled triangle: $\kappa$ off scale ($>10^{18}$)")]
plt.tight_layout()
# Bottom margin for the shared legend
fig.subplots_adjust(bottom=0.26)
fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=11.6, frameon=True,
           framealpha=0.95, bbox_to_anchor=(0.5, 0.01), handlelength=2.2,
           columnspacing=1.6, borderpad=0.6)
for ext in ("pdf", "png"):
    plt.savefig(FIG / f"Fig2.{ext}", dpi=DPI, bbox_inches="tight")
plt.close()
#print("-> Fig2")

[Parallel(n_jobs=22)]: Using backend LokyBackend with 22 concurrent workers.
Model is not converging.  Current: 9382.374310494863 is not greater than 9382.374342039691. Delta is -3.1544828743790276e-05
Model is not converging.  Current: 9382.374331597945 is not greater than 9382.374351289705. Delta is -1.9691760826390237e-05
Model is not converging.  Current: 24630.24289708804 is not greater than 24630.242901985694. Delta is -4.897654434898868e-06
Model is not converging.  Current: 9382.374330984076 is not greater than 9382.374351197484. Delta is -2.0213408788549714e-05
Model is not converging.  Current: 9382.374310891051 is not greater than 9382.37434294523. Delta is -3.205417851859238e-05
[Parallel(n_jobs=22)]: Done  28 tasks      | elapsed:  5.3min
Model is not converging.  Current: 29761.143076381046 is not greater than 29761.148429532343. Delta is -0.005353151296731085
Model is not converging.  Current: 39413.24301140694 is not greater than 39413.24302742113. Delta is -1.601418625


finished in 50.4 min; 288/288 converged

largest per-state RI proportion (%), by feature set and covariance
cov              diag   full
feature_set                 
F2D_nonoverlap   6.66   6.35
F4D_overlap      9.28  13.55
F_filt           7.38   7.26
F_filt_env       7.89   8.87
F_filt_env_conv  8.68   7.11
F_wind           5.36   5.36

grid maximum            13.55%   (base rate 3.36%)
numerically stable max  8.52%
configurations >= 10%   6 of 288, of which 6 singular
grid maximum attained by F4D_overlap, K=9, full covariance, seed 1, state size 610, kappa 2.50e+15

spread across seeds (sd of the maximum, percentage points)
feature_set      cov 
F2D_nonoverlap   diag    0.887
                 full    0.729
F4D_overlap      diag    0.353
                 full    2.963
F_filt           diag    0.581
                 full    0.474
F_filt_env       diag    0.317
                 full    0.997
F_filt_env_conv  diag    0.520
                 full    0.875
F_wind           diag    0.193
 

## 7. Two-state wind-only model — Table 1 and Figure 3

In [7]:
_WIND_CACHE = {}

def _fit_restart(K, seed, Xtr, ltr):
    m = hmmlearn_hmm.GaussianHMM(n_components=K, covariance_type="diag",
                                 n_iter=500, tol=1e-5, random_state=seed)
    m.fit(Xtr, ltr)
    return m.score(Xtr, ltr), m

def fit_wind_hmm(K, seed=SEED, n_restart=30):
    """Fit a wind-only model with K states; restarts run in parallel."""
    cols = ["MaxWind"]
    if "prep" not in _WIND_CACHE:
        scaler = StandardScaler().fit(df.loc[df.split == "train", cols].values)
        Xtr, ltr, _ = make_seqs(df[df.split == "train"], cols, scaler)
        Xall, _, idx_sets = make_seqs(df, cols, scaler)
        _WIND_CACHE["prep"] = (scaler, Xtr, ltr, Xall, idx_sets)
    scaler, Xtr, ltr, Xall, idx_sets = _WIND_CACHE["prep"]

    results = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(_fit_restart)(K, seed + r, Xtr, ltr) for r in range(n_restart))
    best = max(results, key=lambda z: z[0])[1]
    _WIND_CACHE[f"model{K}"] = best

    mu = scaler.inverse_transform(best.means_)[:, 0]
    sd = np.sqrt(scaler.scale_[0] ** 2 * np.array([best.covars_[k, 0, 0] for k in range(K)]))
    order = np.argsort(mu)[::-1]                     # state 0 = highest intensity
    rank = {raw: r for r, raw in enumerate(order)}

    states = np.full(len(df), -1, int); pos = 0
    for idx in idx_sets:
        states[idx] = best.predict(Xall[pos:pos + len(idx)]); pos += len(idx)
    _WIND_CACHE[f"rank{K}"] = rank
    return np.array([rank.get(s, -1) for s in states]), mu[order], sd[order]

def filter_wind_hmm(K):
    """Causal state assignment: argmax of the forward-filtered posterior."""
    scaler, _, _, Xall, idx_sets = _WIND_CACHE["prep"]
    model, rank = _WIND_CACHE[f"model{K}"], _WIND_CACHE[f"rank{K}"]
    log_A, log_pi = np.log(model.transmat_), np.log(model.startprob_)
    states = np.full(len(df), -1, int); pos = 0
    for idx in idx_sets:
        lp = model._compute_log_likelihood(Xall[pos:pos + len(idx)]); pos += len(idx)
        alpha = log_pi + lp[0]
        out = [alpha.argmax()]
        for t in range(1, len(idx)):
            alpha = np.logaddexp.reduce(alpha[:, None] + log_A, axis=0) + lp[t]
            out.append(alpha.argmax())
        states[idx] = [rank[s] for s in out]
    return states

# Step length to the previous record; transition pairs are kept only when exactly 6 h.
_dt_prev = df.groupby("storm_id")["time"].diff().dt.total_seconds() / 3600.0
df["six_hourly_step"] = (np.abs(_dt_prev - 6.0) < 1e-6).values

states2, mu2, sd2 = fit_wind_hmm(2)
df["state"] = states2
df["cls"] = df.state.map({0: "high", 1: "low"})
df["state_causal"] = filter_wind_hmm(2)
df["cls_causal"] = df.state_causal.map({0: "high", 1: "low"})
_both = (df.state >= 0) & (df.state_causal >= 0)
print(f"Viterbi and forward-filtered classes agree on "
      f"{100 * (df.state[_both] == df.state_causal[_both]).mean():.2f}% of assigned observations")
print(f"emission means: high {mu2[0]:.1f} +/- {sd2[0]:.1f} kt, "
      f"low {mu2[1]:.1f} +/- {sd2[1]:.1f} kt")

K2 = dict(emission_mu_high=float(mu2[0]), emission_mu_low=float(mu2[1]),
          emission_sd_high=float(sd2[0]), emission_sd_low=float(sd2[1]),
          n_total=int(len(df)), n_assigned=int((df.state >= 0).sum()),
          n_unassigned=int((df.state < 0).sum()),
          pct_unassigned=float(100 * (df.state < 0).mean()),
          base_rate=float(100 * df.RI.mean()),
          pct_viterbi_filter_agree=float(100 * (df.state[_both] == df.state_causal[_both]).mean()),
          pct_steps_six_hourly=float(100 * df.six_hourly_step[df.groupby("storm_id").cumcount() > 0].mean()))
print(f"total {K2['n_total']:,} | assigned {K2['n_assigned']:,} | "
      f"unassigned {K2['n_unassigned']:,} ({K2['pct_unassigned']:.2f}%)")
print(f"base rate {K2['base_rate']:.3f}%")

# ------------------------------- Table 1 ------------------------------------
rows = []
for code, name in [(0, "high"), (1, "low")]:
    all_obs = df.state == code
    with_outcome = all_obs & df.RI.notna()
    row = dict(cls=name, n=int(all_obs.sum()),
               pct_of_total=float(100 * all_obs.mean()),
               pct_of_assigned=float(100 * all_obs.sum() / K2["n_assigned"]),
               n_with_outcome=int(with_outcome.sum()),
               pct_outcome_censored=float(100 * (1 - with_outcome.sum() / max(all_obs.sum(), 1))),
               mean_V=float(df.loc[all_obs, "MaxWind"].mean()),
               sd_V=float(df.loc[all_obs, "MaxWind"].std()),
               mean_dV24=float(df.loc[all_obs, "dV_24h"].mean()),
               pi_k=float(100 * df.loc[with_outcome, "RI"].mean()),
               pi_k_all_obs_denominator=float(100 * df.loc[all_obs, "RI"].fillna(0).mean()))
    for v in ["sst_C", "shear", "so26chgt", "mpi_gap_obs", "tp", "r700"]:
        if v in df.columns:
            row[f"mean_{v}"] = float(df.loc[all_obs, v].mean())
    rows.append(row)
table1 = pd.DataFrame(rows)
table1.to_csv(OUT / "05_table1_classes.csv", index=False)
print(table1[["cls", "n", "pct_of_total", "pct_of_assigned", "n_with_outcome",
              "pct_outcome_censored", "mean_V", "pi_k",
              "pi_k_all_obs_denominator"]].round(2).to_string(index=False))
for _, r in table1.iterrows():
    print(f"  {r.cls}: pi_k / base rate = {r.pi_k / K2['base_rate']:.2f}")

# --------------------- transition matrix and bootstrap ----------------------
def transition_counts(states, storm_codes, K, six_hourly):
    """Per-storm K x K counts over consecutive records exactly 6 h apart."""
    same = storm_codes[:-1] == storm_codes[1:]
    a, b = states[:-1], states[1:]
    keep = same & (a >= 0) & (b >= 0) & np.asarray(six_hourly)[1:]
    counts = np.zeros((storm_codes.max() + 1, K, K))
    np.add.at(counts, (storm_codes[:-1][keep], a[keep], b[keep]), 1)
    return counts

storm_codes = pd.factorize(df.storm_id)[0]
counts2 = transition_counts(df.state.values, storm_codes, 2, df.six_hourly_step.values)
total = counts2.sum(0)
A = total / total.sum(1, keepdims=True)

rng = np.random.default_rng(SEED)
n_storms = counts2.shape[0]
boot = np.stack([(lambda T: T / T.sum(1, keepdims=True))(
    counts2[rng.integers(0, n_storms, n_storms)].sum(0)) for _ in range(N_BOOT_STORM)])
lo, hi = np.percentile(boot, 2.5, 0), np.percentile(boot, 97.5, 0)

names = ["high", "low"]
trans_rows = []
for i in range(2):
    for j in range(2):
        trans_rows.append(dict(frm=names[i], to=names[j], p=float(A[i, j]),
                               lo=float(lo[i, j]), hi=float(hi[i, j])))
        print(f"  P({names[i]} -> {names[j]}) = {A[i, j]:.4f} [{lo[i, j]:.4f}, {hi[i, j]:.4f}]")
cross_boot = boot[:, 0, 1] + boot[:, 1, 0]
K2["cross"] = float(A[0, 1] + A[1, 0])
K2["cross_ci"] = [float(np.percentile(cross_boot, 2.5)), float(np.percentile(cross_boot, 97.5))]
print(f"  cross = {K2['cross']:.4f} [{K2['cross_ci'][0]:.4f}, {K2['cross_ci'][1]:.4f}]")
pd.DataFrame(trans_rows).to_csv(OUT / "05_transition_matrix.csv", index=False)

# ------------------------------- dwell times --------------------------------
def dwell_hours(frame, col, K):
    """Time from the first record of a run to the first record after it
    (or to the last record plus 6 h when the run ends the track)."""
    runs = {k: [] for k in range(K)}
    for _, g in frame.groupby("storm_id"):
        v = g[col].values
        t = g["time"].values
        start = 0
        for i in range(1, len(v) + 1):
            if i == len(v) or v[i] != v[start]:
                if v[start] in runs:
                    end = t[i] if i < len(v) else t[i - 1] + np.timedelta64(6, "h")
                    runs[v[start]].append((end - t[start]) / np.timedelta64(1, "h"))
                start = i
    return runs

runs = dwell_hours(df[df.state >= 0], "state", 2)
dwell_rows = []
for code, name in [(0, "high"), (1, "low")]:
    r = np.array(runs[code])
    dwell_rows.append(dict(cls=name, n_runs=len(r), median_h=float(np.median(r)),
                           mean_h=float(r.mean()), p90_h=float(np.percentile(r, 90))))
    K2[f"dwell_median_{name}"] = float(np.median(r))
pd.DataFrame(dwell_rows).to_csv(OUT / "05_dwell.csv", index=False)
print(pd.DataFrame(dwell_rows).round(1).to_string(index=False))
json.dump(K2, open(OUT / "05_k2_summary.json", "w"), indent=2)

# ------------------------------- Figure 3 -----------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
im = ax[0].imshow(A, cmap="Purples", vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, f"{A[i, j]:.3f}\n[{lo[i, j]:.3f}, {hi[i, j]:.3f}]",
                   ha="center", va="center", fontsize=10,
                   color="white" if A[i, j] > 0.5 else "black")
ax[0].set_xticks([0, 1]); ax[0].set_yticks([0, 1])
ax[0].set_xticklabels([r"$\rightarrow$ high", r"$\rightarrow$ low"])
ax[0].set_yticklabels([r"high $\rightarrow$", r"low $\rightarrow$"])
ax[0].set_title("(a) 6-hourly transition matrix\n(95% storm-bootstrap interval)")
ax[0].grid(False)
plt.colorbar(im, ax=ax[0], fraction=0.04, pad=0.04)
for code, name, colour in [(0, "high-intensity", COL_HIGH), (1, "low-intensity", COL_LOW)]:
    r = np.sort(np.array(runs[code]))
    ax[1].step(r, 1.0 - np.arange(len(r)) / len(r), where="post", color=colour, lw=2,
               label=f"{name} (n={len(r)}, median {int(np.median(r))} h)")
ax[1].set_xscale("log")
ax[1].set_xlabel("Dwell time (hours)")
ax[1].set_ylabel("Survival, $1-$CDF")
ax[1].set_title("(b) Dwell-time distributions")
ax[1].legend(loc="upper right", fontsize=8.5)
plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(FIG / f"Fig3.{ext}", dpi=DPI, bbox_inches="tight")
plt.close()
print("-> Fig3")


Viterbi and forward-filtered classes agree on 94.98% of assigned observations
emission means: high 76.4 +/- 13.8 kt, low 44.7 +/- 7.1 kt
total 28,156 | assigned 27,910 | unassigned 246 (0.87%)
base rate 3.364%
 cls     n  pct_of_total  pct_of_assigned  n_with_outcome  pct_outcome_censored  mean_V  pi_k  pi_k_all_obs_denominator
high 13415         47.65            48.07           11636                 13.26   77.21  2.32                      2.01
 low 14495         51.48            51.93           10239                 29.36   44.30  4.55                      3.21
  high: pi_k / base rate = 0.69
  low: pi_k / base rate = 1.35
  P(high -> high) = 0.9572 [0.9539, 0.9603]
  P(high -> low) = 0.0428 [0.0397, 0.0461]
  P(low -> high) = 0.0532 [0.0497, 0.0568]
  P(low -> low) = 0.9468 [0.9432, 0.9503]
  cross = 0.0960 [0.0915, 0.1003]
 cls  n_runs  median_h  mean_h  p90_h
high     673      96.0   103.7  182.4
 low    1659      42.0    49.9   96.0
-> Fig3


## 8. Intensity threshold and conditional tests — Tables 2-3 and Figure 4

In [8]:
# -------------------------- overlap between classes -------------------------
overlap = (df[df.state >= 0].groupby(["MaxWind", "cls"]).size()
             .unstack(fill_value=0).reindex(columns=["high", "low"], fill_value=0))
overlap["pct_high"] = 100 * overlap.high / (overlap.high + overlap.low)
overlap.to_csv(OUT / "06_threshold_overlap.csv")
mixed = overlap[(overlap.high > 0) & (overlap.low > 0)]
minority = int(np.minimum(overlap.high, overlap.low).sum())
assigned = int((overlap.high + overlap.low).sum())

THR = dict(overlap_min_kt=float(mixed.index.min()) if len(mixed) else None,
           overlap_max_kt=float(mixed.index.max()) if len(mixed) else None,
           minority_n=minority, n_assigned=assigned,
           minority_pct=float(100 * minority / assigned),
           implied_threshold_kt=float(overlap.index[(overlap.pct_high >= 50).argmax()]))
json.dump(THR, open(OUT / "06_threshold.json", "w"), indent=2)
print("wind speeds at which both classes occur:")
print(mixed.round(1).to_string())
print(f"\nimplied threshold ~{THR['implied_threshold_kt']:.0f} kt; "
      f"{minority:,} of {assigned:,} assigned observations ({THR['minority_pct']:.2f}%) "
      f"fall in the minority class at their own wind speed")

# ------------------------ stratified test machinery -------------------------
def strat_table(y, flag, strata):
    """Collapse to per-stratum 2x2 counts."""
    codes, _ = pd.factorize(pd.Series(strata).astype(str).fillna("NA").values, sort=False)
    S = codes.max() + 1
    n = np.bincount(codes, minlength=S).astype(np.int64)
    m = np.bincount(codes, weights=flag, minlength=S).astype(np.int64)
    k = np.bincount(codes, weights=y, minlength=S).astype(np.int64)
    a = np.bincount(codes, weights=y * flag, minlength=S).astype(np.int64)
    return n, m, k, a

def cmh_test(n, m, k, a, n_perm=N_PERM, seed=SEED):
    """Exact stratified permutation test, CMH statistic and Mantel-Haenszel OR."""
    rng = np.random.default_rng(seed)
    informative = (m > 0) & (m < n) & (k > 0) & (k < n) & (n > 1)
    T, T_fixed = a.sum(), a[~informative].sum()
    ni, mi, ki, ai = (n[informative].astype(float), m[informative].astype(float),
                      k[informative].astype(float), a[informative].astype(float))
    if len(ni) == 0:
        return dict(status="degenerate", n_strata=int(len(n)), n_info=0, info_frac=0.0,
                    p_perm=np.nan, p_cmh=np.nan, or_mh=np.nan, or_lo=np.nan,
                    or_hi=np.nan, rd_mh=np.nan, T_obs=int(T), E_T=np.nan, Z=np.nan)

    E = (mi * ki / ni).sum() + T_fixed
    V = (mi * (ni - mi) * ki * (ni - ki) / (ni ** 2 * (ni - 1))).sum()
    Z = (T - E) / np.sqrt(V) if V > 0 else np.nan
    chi2 = (abs(T - E) - 0.5) ** 2 / V if V > 0 else np.nan
    p_cmh = float(sps.chi2.sf(chi2, 1)) if np.isfinite(chi2) else np.nan

    chunk = max(200, int(4e6 / len(ni)))
    draws = np.empty(n_perm, np.int64)
    for start in range(0, n_perm, chunk):
        end = min(start + chunk, n_perm)
        draws[start:end] = rng.hypergeometric(
            ki.astype(int)[:, None], (ni - ki).astype(int)[:, None],
            mi.astype(int)[:, None], size=(len(ni), end - start)).sum(0) + T_fixed
    p_perm = max(float((np.abs(draws - E) >= abs(T - E) - 1e-9).mean()), 1.0 / n_perm)

    aa, bb, cc, dd = ai, mi - ai, ki - ai, (ni - mi) - (ki - ai)
    R, S_ = (aa * dd / ni).sum(), (bb * cc / ni).sum()
    if R > 0 and S_ > 0:
        or_mh = R / S_
        P, Q_ = (aa + dd) / ni, (bb + cc) / ni
        Rs, Ss = aa * dd / ni, bb * cc / ni
        var_ln = ((P * Rs).sum() / (2 * R ** 2)
                  + (P * Ss + Q_ * Rs).sum() / (2 * R * S_)
                  + (Q_ * Ss).sum() / (2 * S_ ** 2))
        se = np.sqrt(var_ln)
        or_lo, or_hi = or_mh * np.exp(-1.96 * se), or_mh * np.exp(1.96 * se)
    else:
        or_mh = or_lo = or_hi = np.nan

    w = mi * (ni - mi) / ni
    rd = aa / mi - cc / (ni - mi)
    return dict(status="ok", n_strata=int(len(n)), n_info=int(informative.sum()),
                info_frac=float(ni.sum() / n.sum()), p_perm=p_perm, p_cmh=p_cmh,
                or_mh=float(or_mh), or_lo=float(or_lo), or_hi=float(or_hi),
                rd_mh=float((w * rd).sum() / w.sum()), T_obs=int(T), E_T=float(E), Z=float(Z))

def boot_rd(y, flag, strata, storm, n_boot=N_BOOT_STORM, seed=SEED + 7):
    """Storm-level block bootstrap of the stratified risk difference."""
    rng = np.random.default_rng(seed)
    codes, _ = pd.factorize(storm)
    n_storms = codes.max() + 1
    by_storm = [np.where(codes == i)[0] for i in range(n_storms)]
    strat_codes, _ = pd.factorize(pd.Series(strata).astype(str).fillna("NA").values, sort=False)
    out = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.concatenate([by_storm[i] for i in rng.integers(0, n_storms, n_storms)])
        yy, ff, ss = y[idx], flag[idx], strat_codes[idx]
        S = ss.max() + 1
        n_ = np.bincount(ss, minlength=S).astype(float)
        m_ = np.bincount(ss, weights=ff, minlength=S)
        k_ = np.bincount(ss, weights=yy, minlength=S)
        a_ = np.bincount(ss, weights=yy * ff, minlength=S)
        ok_ = (m_ > 0) & (m_ < n_)
        if not ok_.any():
            out[b] = np.nan; continue
        w = m_[ok_] * (n_[ok_] - m_[ok_]) / n_[ok_]
        out[b] = (w * (a_[ok_] / m_[ok_] - (k_[ok_] - a_[ok_]) / (n_[ok_] - m_[ok_]))).sum() / w.sum()
    out = out[np.isfinite(out)]
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

# ------------------------------ indicators ----------------------------------
# Class and wind at t - 6 h, matched on timestamp.
_code = {0.0: "high", 1.0: "low"}
df["cls_prev"] = pd.Series(val_at_offset(df.assign(_s=df.state.where(df.state >= 0).astype(float)),
                                         6, col="_s")).map(_code).values
df["cls_causal_prev"] = pd.Series(val_at_offset(
    df.assign(_s=df.state_causal.where(df.state_causal >= 0).astype(float)),
    6, col="_s")).map(_code).values
df["flag_membership"] = (df.cls == "high").astype(float)
df["flag_transition"] = ((df.cls_prev == "low") & (df.cls == "high")).astype(float)
df["flag_transition_causal"] = ((df.cls_causal_prev == "low")
                                & (df.cls_causal == "high")).astype(float)
df["flag_threshold"] = ((df.MaxWind >= 60) & (df.V_m6 < 60)).astype(float)

D = df[df.RI.notna() & df.dV_12h.notna() & df.cls.notna() & df.cls_prev.notna()
       & df.cls_causal_prev.notna()].copy()

# How the model indicators relate to the fixed rule
_only_thr = (D.flag_threshold == 1) & (D.flag_transition == 0)
_only_hmm = (D.flag_transition == 1) & (D.flag_threshold == 0)
FLAGS = dict(n_test=int(len(D)),
             n_transition=int(D.flag_transition.sum()),
             n_transition_causal=int(D.flag_transition_causal.sum()),
             n_threshold=int(D.flag_threshold.sum()),
             pct_agree_transition_threshold=float(100 * (D.flag_transition == D.flag_threshold).mean()),
             pct_agree_causal_threshold=float(100 * (D.flag_transition_causal == D.flag_threshold).mean()),
             n_threshold_only=int(_only_thr.sum()),
             n_threshold_only_RI=int(D.RI[_only_thr].sum()),
             mean_future_dV24_threshold_only=float(D.loc[_only_thr, "dV_24h_future"].mean()),
             n_transition_only=int(_only_hmm.sum()))
json.dump(FLAGS, open(OUT / "06_indicator_overlap.json", "w"), indent=2)
print(json.dumps(FLAGS, indent=2))
D["shear_q"] = pd.qcut(D.shear, 4, labels=False, duplicates="drop")
D["sst_q"]   = pd.qcut(D.sst_C, 4, labels=False, duplicates="drop")
y = D.RI.values.astype(np.int64)
storm = D.storm_id.values
V   = D.MaxWind.round(0).astype(int).astype(str)
d12 = D.dV_12h.round(0).astype(int).astype(str)
d24 = D.dV_24h.fillna(-999).round(0).astype(int).astype(str)

SCHEMES = {
    "none":                  pd.Series(np.zeros(len(D), int), index=D.index).astype(str),
    "Vt":                    V,
    "Vt_x_dV12":              V + "|" + d12,
    "Vt_x_dV12_x_dV24":      V + "|" + d12 + "|" + d24,
    "Vt_x_dV12_x_shear_sst": V + "|" + d12 + "|" + D.shear_q.fillna(-1).astype(int).astype(str)
                             + "|" + D.sst_q.fillna(-1).astype(int).astype(str),
}
FLAG_LABEL = {"flag_membership":        "High-intensity class membership",
              "flag_transition":        "Class transition, Viterbi (uses future)",
              "flag_transition_causal": "Class transition, causal filter",
              "flag_threshold":         "Fixed 60-kt threshold crossing"}
SCHEME_LABEL = {"none": "Unconditional", "Vt": r"| $V_t$",
                "Vt_x_dV12": r"| $V_t$, $\Delta V_{12}$",
                "Vt_x_dV12_x_dV24": r"| $V_t$, $\Delta V_{12}$, $\Delta V_{24}$",
                "Vt_x_dV12_x_shear_sst": r"| $V_t$, $\Delta V_{12}$, shear, SST"}

print(f"\ntest sample n={len(D):,}, storms {D.storm_id.nunique():,}, "
      f"RI base {100 * y.mean():.2f}%")
PERM = []
for flag_name in ["flag_membership", "flag_transition", "flag_transition_causal",
                  "flag_threshold"]:
    f = D[flag_name].values.astype(np.int64)
    print(f"\n{FLAG_LABEL[flag_name]}: n(flag=1)={int(f.sum()):,}  "
          f"RI|1={100 * y[f == 1].mean():.2f}%  RI|0={100 * y[f == 0].mean():.2f}%")
    for scheme, strata in SCHEMES.items():
        res = cmh_test(*strat_table(y, f, strata))
        if res["status"] == "ok" and res["info_frac"] >= 0.02:
            rd_lo, rd_hi = boot_rd(y, f, strata, storm)
        else:
            rd_lo = rd_hi = np.nan
        res.update(flag=flag_name, flag_label=FLAG_LABEL[flag_name],
                   scheme=scheme, scheme_label=SCHEME_LABEL[scheme],
                   rd_lo=rd_lo, rd_hi=rd_hi)
        PERM.append(res)
        print(f"  {scheme:24s} info={100 * res['info_frac']:5.1f}%  "
              f"OR={res['or_mh']:6.2f} [{res['or_lo']:.2f}, {res['or_hi']:.2f}]  "
              f"p={res['p_perm']:.4g}  RD=[{100 * rd_lo:+.2f}, {100 * rd_hi:+.2f}] pp")
PERM = pd.DataFrame(PERM)
PERM.to_csv(OUT / "06_conditional_tests.csv", index=False)

# ------------------------------- Figure 4 -----------------------------------
# Larger fonts for publication
plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11.5
})

fig, ax = plt.subplots(1, 2, figsize=(13.5, 5.6))

ax[0].bar(overlap.index, overlap.low, width=4.2, color=COL_LOW,
          edgecolor="black", lw=0.4, label="Low-intensity class")
ax[0].bar(overlap.index, overlap.high, width=4.2, bottom=overlap.low, color=COL_HIGH,
          edgecolor="black", lw=0.4, label="High-intensity class")
ax[0].set_xlabel("Maximum sustained wind (kt)")
ax[0].set_ylabel("Number of observations")
ax[0].set_title("(a) The $K$=2 partition is a hard intensity threshold\n"
                f"({THR['minority_pct']:.2f}% of observations in the minority class)")
ax[0].legend(fontsize=11.5)

order = list(SCHEMES.keys())
xs = np.arange(len(order))
offsets = {"flag_membership": -0.27, "flag_transition": -0.09,
           "flag_transition_causal": 0.09, "flag_threshold": 0.27}
colours = {"flag_membership": "#C0392B", "flag_transition": "#8E44AD",
           "flag_transition_causal": "#16A085", "flag_threshold": "#7F8C8D"}
markers = {"flag_membership": "o", "flag_transition": "s",
           "flag_transition_causal": "D", "flag_threshold": "^"}
for flag_name in offsets:
    s = PERM[PERM.flag == flag_name].set_index("scheme").reindex(order)
    lo_err = (s.or_mh - s.or_lo).clip(lower=0)
    hi_err = (s.or_hi - s.or_mh).clip(lower=0)
    ax[1].errorbar(xs + offsets[flag_name], s.or_mh, yerr=[lo_err, hi_err],
                   fmt=markers[flag_name],
                   ms=7, capsize=3, lw=1.4, color=colours[flag_name],
                   label=FLAG_LABEL[flag_name])
ax[1].axhline(1, color="black", ls="--", lw=1.1)
ax[1].set_xticks(xs)
ax[1].set_xticklabels([SCHEME_LABEL[k] for k in order], rotation=18, ha="right", fontsize=11)
ax[1].set_yscale("log")
ax[1].set_ylabel("Mantel--Haenszel odds ratio for RI")
ax[1].set_xlabel("Conditioning set")
ax[1].set_title("(b) Information beyond intensity and persistence")
ax[1].legend(fontsize=10.5, loc="upper left")

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(FIG / f"Fig4.{ext}", dpi=DPI, bbox_inches="tight")
plt.close()
print("\n-> Fig4")

wind speeds at which both classes occur:
cls      high   low  pct_high
MaxWind                      
50         37  2747       1.3
55        117  2121       5.2
60       1888   193      90.7
65       1766     6      99.7

implied threshold ~60 kt; 353 of 27,910 assigned observations (1.26%) fall in the minority class at their own wind speed
{
  "n_test": 19198,
  "n_transition": 670,
  "n_transition_causal": 708,
  "n_threshold": 759,
  "pct_agree_transition_threshold": 99.53641004271279,
  "pct_agree_causal_threshold": 94.47338264402542,
  "n_threshold_only": 89,
  "n_threshold_only_RI": 0,
  "mean_future_dV24_threshold_only": -6.235955056179775,
  "n_transition_only": 0
}

test sample n=19,198, storms 1,072, RI base 2.95%

High-intensity class membership: n(flag=1)=11,317  RI|1=2.39%  RI|0=3.76%
  none                     info=100.0%  OR=  0.63 [0.53, 0.74]  p=2e-05  RD=[-2.00, -0.77] pp
  Vt                       info= 34.1%  OR=  3.21 [1.16, 8.94]  p=0.08798  RD=[+1.31, +4.34] pp
 

## 9. Seasonal and inter-annual composition — Figure 5

In [9]:
monthly = df.groupby("month").cls.value_counts(normalize=True).unstack().fillna(0) * 100
annual  = df.groupby("year").cls.value_counts(normalize=True).unstack().fillna(0) * 100
monthly.to_csv(OUT / "07_monthly_composition.csv")
annual.to_csv(OUT / "07_annual_composition.csv")
high_share = np.asarray(annual.get("high", pd.Series(index=annual.index, data=0.0)), float)
slope, _, _, p_trend, _ = sps.linregress(np.asarray(annual.index, float), high_share)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].bar(monthly.index, monthly.get("low", 0), color=COL_LOW, label="Low-intensity class")
ax[0].bar(monthly.index, monthly.get("high", 0), bottom=monthly.get("low", 0),
          color=COL_HIGH, label="High-intensity class")
ax[0].set_xlabel("Month"); ax[0].set_ylabel("Share of observations (%)")
ax[0].set_title("(a) Monthly class composition"); ax[0].legend(fontsize=8.5)
ax[1].plot(annual.index, high_share, "o-", color=COL_HIGH, ms=4, lw=1.4)
ax[1].axvline(TRAIN_YEAR_MAX, color="black", ls="--", lw=1)
ax[1].set_xlabel("Year"); ax[1].set_ylabel("High-intensity class share (%)")
ax[1].set_title(f"(b) Annual composition (trend {slope:+.3f} %/yr, $p$={p_trend:.2f})")
plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(FIG / f"Fig5.{ext}", dpi=DPI, bbox_inches="tight")
plt.close()
print("-> Fig5")


-> Fig5


## 10. Case studies — Figure 6

In [10]:
plt.rcParams.update({"font.size": 13, "axes.labelsize": 13, "axes.titlesize": 13,
                     "xtick.labelsize": 12, "ytick.labelsize": 12})

CASES = [("HAIYAN", 2013), ("SURIGAE", 2021), ("TRAMI", 2018)]
case_rows = []
fig, axes = plt.subplots(3, 1, figsize=(10, 9.8))
for i, (ax_, (name, year)) in enumerate(zip(axes, CASES)):
    hits = df[df.storm_name.astype(str).str.upper().str.contains(name, na=False)
              & (df.Year == year)]
    if hits.empty:
        ax_.text(0.5, 0.5, f"{name.title()} ({year}): no matching track", ha="center",
                 va="center", transform=ax_.transAxes, fontsize=13)
        ax_.set_axis_off(); continue
    sid = hits.storm_id.value_counts().idxmax()
    g = df[df.storm_id == sid].reset_index(drop=True)
    hours = ((g.time - g.time.iloc[0]).dt.total_seconds() / 3600.0).values
    ax_.plot(hours, g.MaxWind, "k-", lw=2, zorder=3)
    for j in range(len(g) - 1):
        ax_.axvspan(hours[j], hours[j + 1],
                    color=COL_HIGH if g.cls.iloc[j] == "high" else COL_LOW,
                    alpha=0.25, lw=0)
    ri_mask = (g.RI == 1).values
    ax_.plot(hours[ri_mask], g.MaxWind[ri_mask], "r*", ms=13, zorder=4,
             label=f"RI-satisfying records (n={int(ri_mask.sum())})")
    ax_.set_ylabel("Maximum wind (kt)")
    ax_.set_xlabel("Hours since first 35-kt record")
    ax_.set_title(f"({chr(97 + i)}) {name.title()} ({year})", loc="left")
    ax_.legend(fontsize=12, loc="upper right")
    first_high = hours[(g.cls == "high").values]
    case_rows.append(dict(storm=name.title(), year=year, storm_id=sid, n_records=len(g),
                          peak_kt=float(g.MaxWind.max()),
                          hours_to_peak=float(hours[g.MaxWind.values.argmax()]),
                          first_high_class_h=float(first_high[0]) if len(first_high) else np.nan,
                          n_RI_records=int(ri_mask.sum()),
                          RI_hours=json.dumps([round(float(h), 1) for h in hours[ri_mask]]),
                          RI_winds=json.dumps([int(v) for v in g.MaxWind[ri_mask]])))
plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(FIG / f"Fig6.{ext}", dpi=DPI, bbox_inches="tight")
plt.close()
cases = pd.DataFrame(case_rows)
cases.to_csv(OUT / "13_case_studies.csv", index=False)
print(cases.to_string(index=False))
print("-> Fig6")


  storm  year  storm_id  n_records  peak_kt  hours_to_peak  first_high_class_h  n_RI_records                             RI_hours                 RI_winds
 Haiyan  2013 1330_2013         29    125.0           84.0                30.0             6 [18.0, 24.0, 30.0, 36.0, 42.0, 48.0] [45, 55, 60, 65, 75, 85]
Surigae  2021 2102_2021         45    120.0           96.0                42.0             1                               [72.0]                     [90]
  Trami  2018 1824_2018         52    105.0           84.0                30.0             0                                   []                       []
-> Fig6


## 11. Environmental separation — Table 5 and Figure 7

In [11]:
ENV_VARS = [("sst_C", "SST (°C)"), ("shear", "VWS 850\u2013200 hPa (kt)"),
            ("so26chgt", r"$D_{26}$ (m)"), ("mpi_gap_obs", "MPI gap (kt)"),
            ("tp", "Total precipitation (m)"), ("r700", "RH 700 hPa (%)"),
            ("vo850", r"Vorticity 850 hPa (s$^{-1}$)"), ("sohtc300", r"OHC 300 m (J m$^{-2}$)")]
ENV_VARS = [(v, l) for v, l in ENV_VARS if v in df.columns]

rows = []
for var, label in ENV_VARS:
    hi_ = df.loc[df.cls == "high", var].dropna()
    lo_ = df.loc[df.cls == "low", var].dropna()
    ri_ = df.loc[df.RI == 1, var].dropna()
    no_ = df.loc[df.RI == 0, var].dropna()
    ks_cls, ks_ri = sps.ks_2samp(hi_, lo_), sps.ks_2samp(ri_, no_)
    rows.append(dict(var=var, label=label, mean_high=hi_.mean(), mean_low=lo_.mean(),
                     D_class=ks_cls.statistic, p_class=ks_cls.pvalue,
                     mean_RI=ri_.mean(), mean_nonRI=no_.mean(),
                     D_RI=ks_ri.statistic, p_RI=ks_ri.pvalue))
env_table = pd.DataFrame(rows)
env_table.to_csv(OUT / "08_env_ks.csv", index=False)
print(env_table.round(4).to_string(index=False))

plt.rcParams.update({"font.size": 16, "axes.labelsize": 15, "axes.titlesize": 16,
                     "xtick.labelsize": 13, "ytick.labelsize": 13, "legend.fontsize": 14})
fig, axes = plt.subplots(2, 4, figsize=(17, 9))
DISPLAY = {"tp": (1e3, "Total precipitation (mm)"),
           "vo850": (1e4, r"Vorticity 850 hPa (10$^{-4}$ s$^{-1}$)"),
           "sohtc300": (1e-9, r"OHC 300 m (GJ m$^{-2}$)")}
for i, (var, label) in enumerate(ENV_VARS[:8]):
    A_ = axes.ravel()[i]
    scale, label = DISPLAY.get(var, (1.0, label))
    hi_ = df.loc[df.cls == "high", var].dropna() * scale
    lo_ = df.loc[df.cls == "low", var].dropna() * scale
    lo_lim, hi_lim = np.nanpercentile(pd.concat([hi_, lo_]), [0.5, 99.5])
    bins = np.linspace(lo_lim, hi_lim, 50)
    A_.hist(lo_, bins=bins, color=COL_LOW, alpha=0.6, density=True, label="low")
    A_.hist(hi_, bins=bins, color=COL_HIGH, alpha=0.6, density=True, label="high")
    row_ = env_table.loc[env_table["var"] == var].iloc[0]
    A_.set_title(f"({chr(97 + i)}) {label}\n"
                 f"$D_{{class}}$ = {row_.D_class:.3f}, $D_{{RI}}$ = {row_.D_RI:.3f}",
                 fontsize=14)
    A_.set_xlabel(label, fontsize=14)
    A_.set_ylabel("Probability density", fontsize=13)
    if i == 0:
        A_.legend(fontsize=13)

for j in range(len(ENV_VARS), 8):
    axes.ravel()[j].axis("off")

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(FIG / f"Fig7.{ext}", dpi=DPI, bbox_inches="tight")
plt.close()
print("-> Fig7")

        var                        label     mean_high      mean_low  D_class  p_class       mean_RI    mean_nonRI   D_RI   p_RI
      sst_C                     SST (°C)  2.738250e+01  2.720000e+01   0.0466      0.0  2.878080e+01  2.779640e+01 0.3547 0.0000
      shear         VWS 850–200 hPa (kt)  2.660850e+01  2.241070e+01   0.1034      0.0  1.854150e+01  2.310810e+01 0.1356 0.0000
   so26chgt                 $D_{26}$ (m)  5.985140e+01  5.902110e+01   0.0570      0.0  9.473680e+01  6.510760e+01 0.4163 0.0000
mpi_gap_obs                 MPI gap (kt) -5.852050e+01 -2.552420e+01   0.9750      0.0 -3.449780e+01 -4.405220e+01 0.2544 0.0000
         tp      Total precipitation (m)  4.300000e-03  2.700000e-03   0.1608      0.0  3.300000e-03  3.500000e-03 0.0813 0.0002
       r700               RH 700 hPa (%)  8.689490e+01  8.521310e+01   0.0719      0.0  8.397560e+01  8.589360e+01 0.1251 0.0000
      vo850 Vorticity 850 hPa (s$^{-1}$)  7.000000e-04  3.000000e-04   0.5127      0.0  4.000000e

## 12. Higher-state structure — Figure 8

In [12]:
plt.rcParams.update({"font.size": 13, "axes.labelsize": 13, "axes.titlesize": 13,
                     "xtick.labelsize": 11, "ytick.labelsize": 11})

higher = []
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))
for idx, (ax_, K) in enumerate(zip(axes, [3, 4, 5])):
    states_K, mu_K, _ = fit_wind_hmm(K)
    df[f"state{K}"] = states_K
    counts = transition_counts(states_K, pd.factorize(df.storm_id)[0], K,
                               df.six_hourly_step.values).sum(0)
    A_K = counts / np.maximum(counts.sum(1, keepdims=True), 1)
    half = int(np.ceil(K / 2))            # states 0..half-1 form the upper half
    crossing = counts[:half, half:].sum() + counts[half:, :half].sum()
    higher.append(dict(K=K, means=json.dumps([round(float(x), 1) for x in mu_K]),
                       diag=json.dumps([round(float(A_K[k, k]), 3) for k in range(K)]),
                       matrix=json.dumps(np.round(A_K, 3).tolist()),
                       boundary_crossing_freq=float(crossing / counts.sum())))
    im = ax_.imshow(A_K, cmap="Purples", vmin=0, vmax=1)
    for i in range(K):
        for j in range(K):
            ax_.text(j, i, f"{A_K[i, j]:.2f}", ha="center", va="center", fontsize=11,
                     color="white" if A_K[i, j] > 0.5 else "black")
    ax_.set_xticks(range(K)); ax_.set_yticks(range(K))
    ax_.set_xticklabels([f"{m:.0f}" for m in mu_K], fontsize=11)
    ax_.set_yticklabels([f"{m:.0f}" for m in mu_K], fontsize=11)
    ax_.set_xlabel("To state (mean, kt)")
    if idx == 0:
        ax_.set_ylabel("From state (mean, kt)")
    ax_.set_title(f"({chr(97 + idx)}) $K$={K}", loc="left")
    ax_.grid(False)

# K = 2 reference value on the same definition
_c2 = counts2.sum(0)
K2["boundary_crossing_freq"] = float((_c2[0, 1] + _c2[1, 0]) / _c2.sum())
json.dump(K2, open(OUT / "05_k2_summary.json", "w"), indent=2)

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(FIG / f"Fig8.{ext}", dpi=DPI, bbox_inches="tight")
plt.close()
pd.DataFrame(higher).to_csv(OUT / "09_higher_K.csv", index=False)
print(pd.DataFrame(higher)[["K", "means", "diag", "boundary_crossing_freq"]].to_string(index=False))
print(f"K=2 boundary-crossing frequency: {K2['boundary_crossing_freq']:.4f}")
print("-> Fig8")


Model is not converging.  Current: 227.69717182599044 is not greater than 227.6971779569319. Delta is -6.130941471838014e-06


 K                          means                                diag  boundary_crossing_freq
 3             [82.1, 56.2, 40.3]               [0.941, 0.847, 0.891]                0.062477
 4       [85.7, 61.9, 44.9, 35.0]        [0.937, 0.844, 0.837, 0.647]                0.058404
 5 [88.6, 66.5, 49.4, 40.0, 35.0] [0.933, 0.839, 0.804, 0.567, 0.647]                0.073306
K=2 boundary-crossing frequency: 0.0485
-> Fig8


## 13. Precursor state distribution — Table 4 and Figure 9

In [13]:
plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12
})

lookup = dict(zip(zip(df.storm_id, df.time), zip(df.cls, df.RI)))
onset = np.zeros(len(df), int)
for i, (sid, t, r) in enumerate(zip(df.storm_id, df.time, df.RI)):
    if r == 1:
        prev = lookup.get((sid, t - pd.Timedelta(hours=6)))
        if prev is None or prev[1] != 1:
            onset[i] = 1
df["RI_onset"] = onset
n_onsets = int(onset.sum())
print(f"RI onsets: {n_onsets}")

LAGS = [6, 12, 18, 24]
counts = {l: {"high": 0, "low": 0, "missing": 0} for l in LAGS}
for sid, t in zip(df.storm_id[df.RI_onset == 1], df.time[df.RI_onset == 1]):
    for lag in LAGS:
        rec = lookup.get((sid, t - pd.Timedelta(hours=lag)))
        c = rec[0] if rec is not None else None
        counts[lag][c if c in ("high", "low") else "missing"] += 1
precursor = pd.DataFrame(counts).T.reset_index().rename(columns={"index": "lag_h"})
precursor["total"] = precursor[["high", "low", "missing"]].sum(1)
precursor["pct_high_of_valid"] = 100 * precursor.high / (precursor.high + precursor.low).clip(lower=1)
precursor.to_csv(OUT / "10_precursor.csv", index=False)
print(precursor.round(1).to_string(index=False))
baseline_high = 100 * (df.cls == "high").mean()

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
arr = precursor[["high", "low", "missing"]].values.astype(float)
pct = 100 * arr / arr.sum(1, keepdims=True)
xs = np.arange(len(LAGS)); bottom = np.zeros(len(LAGS))
for i, (name, colour) in enumerate([("high-intensity", COL_HIGH),
                                    ("low-intensity", COL_LOW),
                                    ("no record at lag", "#BDC3C7")]):
    ax[0].bar(xs, pct[:, i], 0.6, bottom=bottom, color=colour,
              edgecolor="black", lw=0.4, label=name)
    for x, v, b in zip(xs, pct[:, i], bottom):
        if v > 4:
            ax[0].text(x, b + v / 2, f"{v:.1f}%", ha="center", va="center", fontsize=12,
                       color="white" if i < 2 else "black")
    bottom += pct[:, i]
ax[0].set_xticks(xs); ax[0].set_xticklabels([f"$t-${l} h" for l in LAGS])
ax[0].set_ylabel(f"Share of RI onsets (n={n_onsets})")
ax[0].set_ylim(0, 105)
ax[0].set_title("(a) Pre-onset class distribution by lag")
ax[0].legend(fontsize=12)

ax[1].plot(LAGS, precursor.pct_high_of_valid, "o-", color=COL_HIGH, lw=2.2, ms=9,
           label="High-intensity share (assigned only)")
ax[1].axhline(baseline_high, color="black", ls="--", lw=1.2,
              label=f"Dataset baseline ({baseline_high:.1f}%)")
ax[1].set_xlabel("Lag before RI onset (hours)")
ax[1].set_ylabel("High-intensity class share (%)")
ax[1].set_xticks(LAGS); ax[1].set_ylim(0, 100); ax[1].invert_xaxis()
ax[1].set_title("(b) High-intensity share among RI precursors")
ax[1].legend(fontsize=12)

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(FIG / f"Fig9.{ext}", dpi=DPI, bbox_inches="tight")
plt.close()
print("-> Fig9")

RI onsets: 292
 lag_h  high  low  missing  total  pct_high_of_valid
     6    50  153       89    292               24.6
    12    35  147      110    292               19.2
    18    24  125      143    292               16.1
    24    19  110      163    292               14.7
-> Fig9


## 14. Supervised baselines — Table 6 and Figure 10

In [14]:
ENV_BASE = [c for c in ["sst_C", "shear", "so26chgt", "mpi_gap_obs"] if c in df.columns]
CONV = [c for c in ["tp", "r700", "r500", "vo850", "sohtc300", "somxl030"] if c in df.columns]

# Long labels wrap at the '+' sign
SETS = {
    "Non-causal smoother\n+ environment":      ["V_smooth", "dV_24h_s"] + ENV_BASE,
    "Causal filter\n+ environment":            ["V_filt", "dV_24h_f"] + ENV_BASE,
    "Raw quantised\n+ environment":            ["MaxWind", "dV_24h"] + ENV_BASE,
    "Raw + environment\n+ convective":         ["MaxWind", "dV_24h"] + ENV_BASE + CONV,
    "Intensity +\n" r"backward $\Delta V_{24h}$":  ["MaxWind", "dV_24h"],
    "Current intensity only":                  ["MaxWind"],
}
NON_CAUSAL = "Non-causal smoother\n+ environment"
all_cols = sorted({c for v in SETS.values() for c in v})
DD = df.dropna(subset=all_cols + ["RI"]).copy()
train, test = DD[DD.split == "train"], DD[DD.split == "test"]
print(f"common sample: train {len(train):,}, test {len(test):,}, "
      f"test base rate {100 * test.RI.mean():.2f}%")

rows = []
for name, cols in SETS.items():
    scaler = StandardScaler().fit(train[cols])
    model = LogisticRegression(max_iter=5000).fit(scaler.transform(train[cols]),
                                                  train.RI.values.astype(int))
    p = model.predict_proba(scaler.transform(test[cols]))[:, 1]
    y_test = test.RI.values.astype(int)
    cv_scores = []
    for tr_i, te_i in GroupKFold(n_splits=5).split(DD[cols], DD.RI, groups=DD.storm_id):
        sc = StandardScaler().fit(DD[cols].iloc[tr_i])
        lr = LogisticRegression(max_iter=5000).fit(sc.transform(DD[cols].iloc[tr_i]),
                                                   DD.RI.iloc[tr_i].astype(int))
        cv_scores.append(roc_auc_score(DD.RI.iloc[te_i].astype(int),
                                       lr.predict_proba(sc.transform(DD[cols].iloc[te_i]))[:, 1]))
    rows.append(dict(feature_set=name, n_features=len(cols),
                     auc_test=roc_auc_score(y_test, p),
                     pr_auc_test=average_precision_score(y_test, p),
                     brier_test=brier_score_loss(y_test, p),
                     base_rate_test=float(y_test.mean()),
                     auc_cv_storm_mean=float(np.mean(cv_scores)),
                     auc_cv_storm_sd=float(np.std(cv_scores))))
    
    name_clean = name.replace("\n", " ")
    print(f"  {name_clean:42s} AUC={rows[-1]['auc_test']:.3f}  PR={rows[-1]['pr_auc_test']:.3f}  "
          f"CV={np.mean(cv_scores):.3f}+/-{np.std(cv_scores):.3f}")

baselines = pd.DataFrame(rows)
baselines.to_csv(OUT / "11_supervised_baselines.csv", index=False)

defined = df.RI.notna()
LEAK = dict(pr_auc_noncausal=float(baselines.iloc[0].pr_auc_test),
            pr_auc_causal=float(baselines.iloc[1].pr_auc_test),
            auc_noncausal=float(baselines.iloc[0].auc_test),
            auc_causal=float(baselines.iloc[1].auc_test),
            corr_smoother_residual=float(np.corrcoef(
                (df.V_smooth - df.MaxWind)[defined], df.dV_24h_future[defined])[0, 1]),
            corr_filter_residual=float(np.corrcoef(
                (df.V_filt - df.MaxWind)[defined], df.dV_24h_future[defined])[0, 1]))
LEAK["pr_auc_ratio"] = LEAK["pr_auc_noncausal"] / max(LEAK["pr_auc_causal"], 1e-9)
json.dump(LEAK, open(OUT / "11_leakage.json", "w"), indent=2)
print("\n" + json.dumps(LEAK, indent=2))

# Larger fonts for publication
plt.rcParams.update({
    'font.size': 16,
    'axes.labelsize': 16,
    'axes.titlesize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14
})


fig, ax = plt.subplots(1, 2, figsize=(15.5, 6.5))
b = baselines.sort_values("pr_auc_test").reset_index(drop=True)
bar_colours = ["#C0392B" if s == NON_CAUSAL else "#8E44AD" for s in b.feature_set]
bars = ax[0].barh(range(len(b)), b.pr_auc_test, color=bar_colours,
                  edgecolor="black", lw=0.5)
for bar, s in zip(bars, b.feature_set):
    if s == NON_CAUSAL:
        bar.set_hatch("//")


ax[0].set_yticks(range(len(b))); ax[0].set_yticklabels(b.feature_set, fontsize=14.5)
ax[0].axvline(b.base_rate_test.iloc[0], color="grey", ls=":", lw=1.3,
              label=f"Base rate ({100 * b.base_rate_test.iloc[0]:.2f}%)")
ax[0].set_xlabel("PR-AUC (2020--2024 held-out test)")
ax[0].set_title("(a) Non-causal smoothing inflates apparent skill")
ax[0].scatter([], [], marker="s", s=70, facecolor="#C0392B", edgecolor="black",
              hatch="//", label="Uses future observations")

ax[0].legend(fontsize=14, loc="lower right")


for i, v in enumerate(b.pr_auc_test):
    ax[0].text(v + 0.012, i, f"{v:.3f}", va="center", fontsize=14)
ax[0].set_xlim(0, max(b.pr_auc_test) * 1.18)

ax[1].errorbar(range(len(b)), b.auc_cv_storm_mean, yerr=b.auc_cv_storm_sd, fmt="o",
                ms=7, capsize=3, color="#16A085")
ax[1].set_xticks(range(len(b)))

ax[1].set_xticklabels(b.feature_set, rotation=32, ha="right", fontsize=13.6)
ax[1].axhline(0.5, color="black", ls="--", lw=1, label="No skill")
ax[1].set_ylabel("ROC-AUC, storm-grouped 5-fold CV")
ax[1].set_title("(b) Storm-grouped cross-validation")

ax[1].legend(fontsize=14)

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(FIG / f"Fig10.{ext}", dpi=DPI, bbox_inches="tight")
plt.close()
print("-> Fig10")

common sample: train 13,213, test 1,299, test base rate 3.16%
  Non-causal smoother + environment          AUC=0.953  PR=0.546  CV=0.967+/-0.009
  Causal filter + environment                AUC=0.834  PR=0.108  CV=0.872+/-0.022
  Raw quantised + environment                AUC=0.828  PR=0.108  CV=0.870+/-0.023
  Raw + environment + convective             AUC=0.826  PR=0.116  CV=0.872+/-0.024
  Intensity + backward $\Delta V_{24h}$      AUC=0.841  PR=0.133  CV=0.824+/-0.024
  Current intensity only                     AUC=0.542  PR=0.033  CV=0.582+/-0.036

{
  "pr_auc_noncausal": 0.5460538986815309,
  "pr_auc_causal": 0.10802093639235752,
  "auc_noncausal": 0.9531583233161425,
  "auc_causal": 0.8339602155957966,
  "corr_smoother_residual": 0.5496090769785478,
  "corr_filter_residual": -0.3759774805424015,
  "pr_auc_ratio": 5.055074663472036
}
-> Fig10


## 15. Epoch sensitivity

In [15]:
EPOCHS = [("1977-1986", 1977, 1986), ("1987-2005", 1987, 2005), ("2006-2024", 2006, 2024)]
rows = []
for label, y0, y1 in EPOCHS:
    sub = df[(df.year >= y0) & (df.year <= y1)]
    outcome = sub[sub.RI.notna()]
    row = dict(epoch=label, n=len(sub), storms=sub.storm_id.nunique(),
               base_rate=float(100 * outcome.RI.mean()) if len(outcome) else np.nan,
               pct_dV6_zero=float(100 * (sub.dV_6h.dropna() == 0).mean()),
               mean_V=float(sub.MaxWind.mean()),
               pct_high=float(100 * (sub.cls == "high").mean()))
    for code, name in [(0, "high"), (1, "low")]:
        mask = (sub.state == code) & sub.RI.notna()
        row[f"pi_{name}"] = float(100 * sub.loc[mask, "RI"].mean()) if mask.sum() else np.nan
    counts = transition_counts(sub.state.values, pd.factorize(sub.storm_id)[0], 2,
                               sub.six_hourly_step.values).sum(0)
    A_e = counts / np.maximum(counts.sum(1, keepdims=True), 1)
    row["cross"] = float(A_e[0, 1] + A_e[1, 0])
    try:
        scaler = StandardScaler().fit(sub[["MaxWind"]].values)
        Xe, le, _ = make_seqs(sub, ["MaxWind"], scaler)
        m = hmmlearn_hmm.GaussianHMM(n_components=2, covariance_type="diag",
                                     n_iter=500, tol=1e-5, random_state=SEED)
        m.fit(Xe, le)
        mu_e = np.sort(scaler.inverse_transform(m.means_)[:, 0])[::-1]
        row["refit_mu_high"], row["refit_mu_low"] = float(mu_e[0]), float(mu_e[1])
    except Exception:
        row["refit_mu_high"] = row["refit_mu_low"] = np.nan
    rows.append(row)
epochs = pd.DataFrame(rows)
epochs.to_csv(OUT / "12_epoch_sensitivity.csv", index=False)
print(epochs.round(2).to_string(index=False))


    epoch     n  storms  base_rate  pct_dV6_zero  mean_V  pct_high  pi_high  pi_low  cross  refit_mu_high  refit_mu_low
1977-1986  5597     260       3.76         56.63   59.09     44.17     3.52    4.01   0.10          75.86         44.97
1987-2005 12293     504       2.15         51.59   60.32     49.88     1.16    3.41   0.09          76.36         45.10
2006-2024 10266     456       4.63         50.34   59.90     46.86     3.20    6.16   0.09          77.59         42.78


## 16. Supplementary diagnostics quoted in the text

In [16]:
SUPP = {}
_mpi = mpi_kt(df["sst_C"] + 273.15, df["T_out_obs"]).dropna()
SUPP.update(mpi_index_kt_min=float(_mpi.min()), mpi_index_kt_max=float(_mpi.max()),
            mpi_index_kt_mean=float(_mpi.mean()), mpi_index_kt_sd=float(_mpi.std()))
_ok = df["mpi_gap_obs"].notna()
SUPP["corr_mpi_gap_obs_with_V"] = float(np.corrcoef(df.loc[_ok, "mpi_gap_obs"],
                                                    df.loc[_ok, "MaxWind"])[0, 1])

_stable = ok.max_cond < KAPPA_STABLE
SUPP["n_runs_stable"] = int(_stable.sum())
SUPP["mean_max_pi_pct_stable"]   = float(ok.loc[_stable, "max_pi_pct"].mean())
SUPP["mean_max_pi_pct_unstable"] = float(ok.loc[~_stable, "max_pi_pct"].mean())
for _fs in FEATURE_SETS:
    _s = ok[(ok.feature_set == _fs) & _stable]
    SUPP[f"max_pi_pct_stable_{_fs}"] = float(_s.max_pi_pct.max()) if len(_s) else None
    for _cov in GRID_COV:
        _u = ok[(ok.feature_set == _fs) & (ok["cov"] == _cov) & ~_stable].K
        SUPP[f"first_unstable_K_{_fs}_{_cov}"] = int(_u.min()) if len(_u) else None
_sd = ok.groupby(["feature_set", "cov", "K"]).max_pi_pct.std()
SUPP["pct_configs_seed_sd_below_1pp"] = float(100 * (_sd < 1).mean())
SUPP["median_seed_sd_pp"] = float(_sd.median())

_assigned = df[df.state >= 0]
SUPP["high_share_pct_of_assigned"] = float(100 * (_assigned.cls == "high").mean())
for _c in ["high", "low"]:
    _lat = _assigned.loc[_assigned.cls == _c, "Lat"]
    SUPP[f"lat_p10_{_c}"], SUPP[f"lat_p90_{_c}"] = float(_lat.quantile(0.1)), float(_lat.quantile(0.9))
_band = pd.cut(_assigned.Lat, [0, 15, 20, 25, 30, 90])
shear_lat = _assigned.groupby([_band, "cls"], observed=True).shear.mean().unstack()
shear_lat.to_csv(OUT / "14_shear_by_latitude.csv")
json.dump(SUPP, open(OUT / "14_supplementary.json", "w"), indent=2)
print(json.dumps(SUPP, indent=2))
print(shear_lat.round(1).to_string())

{
  "mpi_index_kt_min": 14.669724687516123,
  "mpi_index_kt_max": 20.429322133319477,
  "mpi_index_kt_mean": 18.8969987214515,
  "mpi_index_kt_sd": 0.6371212623030854,
  "corr_mpi_gap_obs_with_V": -0.9994817622864425,
  "n_runs_stable": 147,
  "mean_max_pi_pct_stable": 4.824535282796619,
  "mean_max_pi_pct_unstable": 7.0973663408259595,
  "max_pi_pct_stable_F_wind": 5.360333531864205,
  "first_unstable_K_F_wind_diag": null,
  "first_unstable_K_F_wind_full": null,
  "max_pi_pct_stable_F4D_overlap": 4.4021739130434785,
  "first_unstable_K_F4D_overlap_diag": 2,
  "first_unstable_K_F4D_overlap_full": 3,
  "max_pi_pct_stable_F2D_nonoverlap": 6.044857768052517,
  "first_unstable_K_F2D_nonoverlap_diag": 4,
  "first_unstable_K_F2D_nonoverlap_full": 6,
  "max_pi_pct_stable_F_filt": 7.381288199341796,
  "first_unstable_K_F_filt_diag": null,
  "first_unstable_K_F_filt_full": 5,
  "max_pi_pct_stable_F_filt_env": 4.721239477857752,
  "first_unstable_K_F_filt_env_diag": 4,
  "first_unstable_K_F_filt

## 17. Summary and archive

In [17]:
L = []
A = L.append
A("# Reported quantities\n")
A(f"Generated {time.strftime('%Y-%m-%d %H:%M:%S')} | hmmlearn {ENV_INFO['hmmlearn']} | "
  f"seed {SEED} | Q={Q_PROC} kt^2 | R={R_OBS:.4f} kt^2\n")

A("\n## Data\n")
A(f"- Observations (MaxWind >= {MIN_WIND_KT} kt): **{DIAG['obs_working']:,}** "
  f"from **{DIAG['storms_working']:,}** storms")
A(f"- Smallest reported non-missing wind: **{DIAG['min_MaxWind_present']:.0f} kt**")
A(f"- Duplicate timestamps removed: {DIAG['rows_dup_dropped']:,}")
A(f"- Time steps other than 6 h: **{DIAG['pct_step_not_6h']:.2f}%**")
A(f"- RI labels, row offset: n={DIAG['label_rowshift_n']:,}, "
  f"base {DIAG['label_rowshift_base']:.3f}%, {DIAG['n_RI_rowshift']} cases")
A(f"- RI labels, timestamp match: n={DIAG['label_timebased_n']:,}, "
  f"base **{DIAG['label_timebased_base']:.3f}%**, {DIAG['n_RI_timebased']} cases")
A(f"- delta-V24 differs between conventions for **{DIAG['pct_dV24_differs']:.1f}%** of records")
A(f"- train / valid / test = {DIAG['n_train']:,} / {DIAG['n_valid']:,} / {DIAG['n_test']:,}")

A("\n## Quantisation\n")
A(f"- delta-V6 exactly zero: **{QUANT['pct_dV6_exact_zero']:.1f}%**")
A(f"- delta-V24 exactly zero: {QUANT['pct_dV24_exact_zero']:.1f}%")
A(f"- |delta-V6| < 0.5 kt after smoother {QUANT['pct_dV6_near_zero_smoother']:.1f}%, "
  f"after causal filter {QUANT['pct_dV6_near_zero_filter']:.1f}%")
A(f"- K=6 ({QUANT['k6_covariance']} covariance), {QUANT['k6_n_obs_decoded']:,} decoded "
  f"observations: **{QUANT['k6_states_fully_absorbing']}** fully absorbing state(s), "
  f"**{QUANT['k6_pct_obs_in_absorbing']:.1f}%** of decoded observations; largest state "
  f"n={QUANT['k6_largest_state_n']:,} with {100 * QUANT['k6_largest_state_frac_zero']:.1f}% zeros")

A("\n## Configuration grid\n")
A(f"- Configurations: **{len(grid):,}**, converged {(grid.status == 'ok').sum():,}")
A(f"- Grid maximum per-state RI proportion: **{ok.max_pi_pct.max():.2f}%** "
  f"(base rate {base_rate:.2f}%)")
A(f"- Maximum among numerically stable fits (kappa < {KAPPA_STABLE:g}): "
  f"**{pi_stable_max:.2f}%**")
A(f"- Configurations at or above 10%: **{n_over10}**, of which "
  f"**{n_over10_singular}** are numerically singular")
A(f"- Grid maximum attained by {top.feature_set}, K={top.K}, {top['cov']} covariance, "
  f"kappa {top.max_cond:.2e}")
A("\n```\n" + ok.pivot_table(index="feature_set", columns="cov",
                             values="max_pi_pct", aggfunc="max").round(2).to_string() + "\n```")

A("\n## Two-state model\n")
A(f"- Emission means: high **{K2['emission_mu_high']:.1f} +/- {K2['emission_sd_high']:.1f} kt**, "
  f"low **{K2['emission_mu_low']:.1f} +/- {K2['emission_sd_low']:.1f} kt**")
A(f"- Assigned {K2['n_assigned']:,} of {K2['n_total']:,}; "
  f"unassigned {K2['n_unassigned']:,} ({K2['pct_unassigned']:.2f}%)")
A(f"- Base rate **{K2['base_rate']:.3f}%**")
for _, r in table1.iterrows():
    A(f"- pi_{r.cls} = **{r.pi_k:.2f}%** ({r.pi_k / K2['base_rate']:.2f}x base rate); "
      f"outcome censored for {r.pct_outcome_censored:.1f}% of this class")
for r in trans_rows:
    A(f"- P({r['frm']} -> {r['to']}) = {r['p']:.4f} [{r['lo']:.4f}, {r['hi']:.4f}]")
A(f"- cross = **{K2['cross']:.4f}** [{K2['cross_ci'][0]:.4f}, {K2['cross_ci'][1]:.4f}]")
A(f"- Median dwell (timestamps): high **{K2['dwell_median_high']:.0f} h**, "
  f"low **{K2['dwell_median_low']:.0f} h**")
A(f"- Consecutive record pairs exactly 6 h apart: {K2['pct_steps_six_hourly']:.1f}% "
  f"(transition matrix uses these only)")
A(f"- Viterbi and forward-filtered classes agree on {K2['pct_viterbi_filter_agree']:.2f}% "
  f"of assigned observations")
A(f"- Boundary-crossing frequency per 6-h pair: K=2 {K2['boundary_crossing_freq']:.4f}")

A("\n## Intensity threshold\n")
A(f"- Implied threshold ~**{THR['implied_threshold_kt']:.0f} kt**")
A(f"- Minority-class observations: **{THR['minority_n']:,}** of {THR['n_assigned']:,} "
  f"(**{THR['minority_pct']:.2f}%**)")

A("\n## Indicator overlap\n")
for k_, v_ in FLAGS.items():
    A(f"- {k_}: {v_:.2f}" if isinstance(v_, float) else f"- {k_}: {v_}")

A("\n## Case studies\n")
A("\n```\n" + cases.to_string(index=False) + "\n```")

A("\n## Conditional tests\n")
A("\n```\n" + PERM[["flag", "scheme", "n_info", "info_frac", "or_mh", "or_lo", "or_hi",
                    "p_perm", "rd_mh", "rd_lo", "rd_hi"]].round(4).to_string(index=False) + "\n```")

A("\n## Precursor\n")
A(f"- RI onsets: **{n_onsets}**")
A("\n```\n" + precursor.round(1).to_string(index=False) + "\n```")

A("\n## Environmental separation\n")
A("\n```\n" + env_table[["var", "D_class", "D_RI"]].round(4).to_string(index=False) + "\n```")

A("\n## Supervised baselines\n")
A("\n```\n" + baselines.round(4).to_string(index=False) + "\n```")
A(f"- PR-AUC ratio, non-causal to causal: **{LEAK['pr_auc_ratio']:.2f}**")
A(f"- Correlation of smoother residual with forward 24-h change: "
  f"**{LEAK['corr_smoother_residual']:+.3f}**; causal filter "
  f"{LEAK['corr_filter_residual']:+.3f}")

A("\n## Epoch sensitivity\n")
A("\n```\n" + epochs.round(2).to_string(index=False) + "\n```")

A("\n## Higher-state structure\n")
A("\n```\n" + pd.DataFrame(higher).to_string(index=False) + "\n```")

A("\n## Supplementary diagnostics\n")
for k_, v_ in SUPP.items():
    A(f"- {k_}: {v_:.4f}" if isinstance(v_, float) else f"- {k_}: {v_}")

summary = "\n".join(L)
(OUT / "RESULTS_SUMMARY.md").write_text(summary, encoding="utf-8")
print(summary[:3000])

archive = Path("hmm_ri_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(OUT.parent))
print(f"\n\nWrote {archive} ({archive.stat().st_size / 1e6:.1f} MB)")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size / 1024:.0f} KB)")


# Reported quantities

Generated 2026-10-23 01:17:24 | hmmlearn 0.3.3 | seed 42 | Q=1.0 kt^2 | R=2.0833 kt^2


## Data

- Observations (MaxWind >= 35 kt): **28,156** from **1,220** storms
- Smallest reported non-missing wind: **35 kt**
- Duplicate timestamps removed: 92
- Time steps other than 6 h: **19.16%**
- RI labels, row offset: n=23,338, base 3.072%, 717 cases
- RI labels, timestamp match: n=21,876, base **3.364%**, 736 cases
- delta-V24 differs between conventions for **12.3%** of records
- train / valid / test = 22,580 / 3,196 / 2,380

## Quantisation

- delta-V6 exactly zero: **52.2%**
- delta-V24 exactly zero: 17.8%
- |delta-V6| < 0.5 kt after smoother 18.1%, after causal filter 22.9%
- K=6 (full covariance), 21,505 decoded observations: **1** fully absorbing state(s), **20.0%** of decoded observations; largest state n=11,296 with 60.8% zeros

## Configuration grid

- Configurations: **288**, converged 288
- Grid maximum per-state RI proportion: **13.55%** (base rate 3.36%)
-